In [ ]:
from sklearn import preprocessing
from sklearn.metrics import mean_absolute_percentage_error, r2_score, mean_absolute_percentage_error


# For 2D analysis
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from scipy.optimize import curve_fit
from sklearn.preprocessing import MinMaxScaler
# from utils import period2freq, freq2period

# For PCA
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy.optimize import curve_fit

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import csv
import time
import glob
import os

MULTI_NODES = True # True: uses tasksPerNodes; False: uses parallelism
NORMALIZE = False
INITIAL_STAGE = False
DEBUG = True
STORAGE_LIST = ['localssd', 'beegfs', 'tmpfs', 'nfs']

# Datasize in KB
data_size_kb = {'4mb': 4096, '16mb': 16384, '64mb': 65536,
            '256mb': 262144, '512mb': 524288, '1gb': 1048576,
            '5gb': 5242880, '50gb': 52428800, '100gb': 104857600,
            '300gb': 314572800,}

# Key Parameters
WF_PARAMS = ['operation', 'randomOffset', 'transferSize', 
            'aggregateFilesizeMB', 'numTasks', 'parallelism', 'totalTime', 
            'numNodesList', 'numNodes', 'tasksPerNode', 'trMiB', 'storageType',
            'opCount','taskName','taskPID', 'fileName', 'stageOrder']

TARGET_PARAMS = [ "bestStorage" ]
op_dict = {0: "write", 1: "read"}

test_configs = {
    "ddmd_2n_s": { # old data, less I/O intensive
        "SCRIPT_ORDER": "ddmd_script_order",
        "NUM_NODES_LIST": [1, 2, 4 ],
        "ALLOWED_PARALLELISM": [1, 3, 6, 12],
        "exp_data_path": "./ddmd",
        "test_folders": ['ddmd_2n_pfs_small']
    },
    "ddmd_4n_l": { # normalize global # this is for spm paper
        "SCRIPT_ORDER": "ddmd_script_order",
        "NUM_NODES_LIST": [1, 2, 4 ],
        "exp_data_path": "./ddmd",
        "ALLOWED_PARALLELISM": [1, 3, 6, 12],
        "test_folders": ['ddmd_4n_pfs_large']
    },
    # "ddmd_4n_l": { # normalize global, this is for char_io paper
    #     "SCRIPT_ORDER": "ddmd_script_order",
    #     "NUM_NODES_LIST": [1],
    #     "exp_data_path": "./ddmd",
    #     "ALLOWED_PARALLELISM": [1],
    #     "test_folders": ['ddmd_4n_pfs_large']
    # },
    # "1kg": {
    #     "SCRIPT_ORDER": "1kg_script_order",
    #     "NUM_NODES_LIST": [1, 2, 5, 10, 15],
    #     "ALLOWED_PARALLELISM": [1, 2, 5, 20, 30, 60, 150],
    #     "exp_data_path": "./1kgenome/fastflow_tests", 
    #     "test_folders": ['par_6000_10n_nfs_ps300'] 
    # },
    "1kg": {
        "SCRIPT_ORDER": "1kg_script_order",
        "NUM_NODES_LIST": [10],
        "ALLOWED_PARALLELISM": [1],
        "exp_data_path": "./1kgenome/fastflow_tests", 
        "test_folders": ['par_6000_10n_nfs_ps300'] 
    },
    "1kg_2": {
        "SCRIPT_ORDER": "1kg_script_order",
        "NUM_NODES_LIST": [10],
        "ALLOWED_PARALLELISM": [1],
        "exp_data_path": "./1kgenome/spm_tests", 
        "test_folders": ['par_6000_10n_pfs_ps300'] 
    },
    "pyflex_240f": {
        "SCRIPT_ORDER": "pyflextrkr_script_order",
        "NUM_NODES_LIST": [8, 16, 32 ],
        "ALLOWED_PARALLELISM": [1, 8, 15, 30],
        "exp_data_path": "./pyflextrkr",
        "test_folders": ['summer_sam_8n_pfs']
    },
    "pyflex_s9_48f": {
        "SCRIPT_ORDER": "pyflextrkr_s9_script_order",
        "NUM_NODES_LIST": [4],
        "ALLOWED_PARALLELISM": [1, 12],
        "exp_data_path": "./pyflextrkr",
        "test_folders": ['summer_sam_4n_pfs_s9']
    },
    "ptychonn": {
        "SCRIPT_ORDER": "ptychonn_script_order",
        "NUM_NODES_LIST": [1],
        "ALLOWED_PARALLELISM": [1],
        "exp_data_path": "./ptychonn",
        "test_folders": ['ptychonn_212m'] # ['ptychonn_14m', 'ptychonn_212m']
    },
    "montage": {
        "SCRIPT_ORDER": "montage_script_order",
        "NUM_NODES_LIST": [1],
        "ALLOWED_PARALLELISM": [1],
        "exp_data_path": "./montage",
        "test_folders": ['datalife_montage_1']
    },
    "seismology": {
        "SCRIPT_ORDER": "seismology_script_order",
        "NUM_NODES_LIST": [1],
        "ALLOWED_PARALLELISM": [1],
        "exp_data_path": "./seismology",
        "test_folders": ['seis_1n']
    },
    "llm_wf": {
        "SCRIPT_ORDER": "llm_script_order",
        "NUM_NODES_LIST": [1],
        "ALLOWED_PARALLELISM": [1],
        "exp_data_path": "./llm",
        "test_folders": ['llm_wf_2s']
    }
}

# Load experiment data
CURR_WF="ddmd_4n_l" # ddmd_2n_s, ddmd_4n_l, 1kg, pyflex_240f

SCRIPT_ORDER = test_configs[CURR_WF]["SCRIPT_ORDER"]
NUM_NODES_LIST = test_configs[CURR_WF]["NUM_NODES_LIST"]
ALLOWED_PARALLELISM = test_configs[CURR_WF]["ALLOWED_PARALLELISM"]
exp_data_path = test_configs[CURR_WF]["exp_data_path"]
test_folders = test_configs[CURR_WF]["test_folders"]

# test_folders = ['par_3000_1n_pfs_ps300', 'par_6000_1n_pfs_ps300', 
#                 'par_9000_1n_pfs_ps300'] par_3000_10n_shm_ps300

### Test on PFS:
individuals : 16 minutes and 36 seconds elapsed (996 secs).
individuals_merge : 10 minutes and 36 seconds elapsed (636 secs).
sifting : 0 minutes and 40 seconds elapsed (40 secs).
mutation_overlap : 0 minutes and 31 seconds elapsed (31 secs).
frequency : 4 minutes and 8 seconds elapsed (248 secs).
All done : 32 minutes and 31 seconds elapsed (1951 secs).
### I/O Time breakdown on PFS:
Total I/O time per taskName:
 
 individuals (write): 2.193622e-05 (sec)
 individuals_merge (write): 0.0020965512 (sec)
 mutation_overlap (write): 0.0001629908 (sec)
 frequency (write): 0.00037158170000000003 (sec)

 
 individuals (read): 131.75084598488667 (sec)
 individuals_merge (read): 14.15417127 (sec)
 sifting (read): 20.4605579074 (sec)
 mutation_overlap (read): 0.0305207693 (sec)
 frequency (read): 0.0441538143 (sec)
 
Total I/O time per workflow: 166.44290280580668

### Test on NFS
individuals : 3 minutes and 18 seconds elapsed (198 secs).
individuals_merge+sifting : 15 minutes and 29 seconds elapsed (929 secs).
mutation+frequency : 4 minutes and 29 seconds elapsed (269 secs).
All done : 23 minutes and 16 seconds elapsed (1396 secs).
### I/O Time breakdown on NFS
Total I/O time per taskName:
 
 individuals (write): 4.4967663333333335e-05 (sec)
 individuals_merge (write): 0.0005513027000000001 (sec)
 sifting (write): 0.0002396906 (sec)
 mutation_overlap (write): 0.0083436507 (sec)
 frequency (write): 0.11823377850000001 (sec)

 
 individuals (read): 39.60484083507333 (sec)
 individuals_merge (read): 0.40972155 (sec)
 sifting (read): 9.8425757022 (sec)
 mutation_overlap (read): 0.023316663300000002 (sec)
 frequency (read): 0.2121502092 (sec)
Total I/O time per workflow: 50.220018349936666

In [1]:
# My utility functions
import utils.perf_visualize as pv

# Parameter Notes for Datalife:
Each entry in the table represent only one single edge in the workflow. An directed edge connects a **fileName** and a **taskName**, representing data access.

---
- **operation**: The type of I/O operation {0: "write", 1: "read"}, value 1 represents read (e.g. a directed edge edge from a **fileName** to a **taskName**), value 0 represents write (e.g. a directed edge edge from a **taskName** to a **fileName**)
- **randomOffset**: The type of data access pattern { 0: "sequential file access", 1: "random file access"}
- **transferSize**: Average I/O size of the particular I/O operation to a file, calculated from aggregateFilesizeMB/opCount
- **aggregateFilesizeMB**: Total I/O size of a particular I/O operation to a file for a task
- **numTasks**: Number of parallel tasks for this particular task
- **totalTime**: The total I/O time of of a particular I/O operation to a file for a task
- **numNodes**: Number of nodes used for this particular task
- **tasksPerNode**: numTasks/numNodes for a task
- **bwMiB**: transferRate of a particular I/O operation to a file for a task, calculated from aggregateFilesizeMB/totalTime
- **storageType**: The storage type used in this task. {0: "localssd", 1: "beegfs/pfs", 2: "lustre", 3: "unknown"}
- **opCount**: the number of I/O operation count of a particular I/O operation to a file for a task
- **taskName**: the task name that is running for a particular workflow
- **taskPID**: the task PID
- **fileName**: the name of file that a I/O operation is for

In [2]:
def transform_store_code(storage_type):
    if storage_type == "localssd":
        store_code = 0
    elif storage_type == "beegfs":
        store_code = 1
    elif storage_type == "lustre":
        store_code = 2
    elif storage_type == "tmpfs":
        store_code = 3
    elif storage_type == "nfs":
        store_code = 4
    else:
        store_code = 5
    return store_code

def decode_store_code(store_code):
    mapping = {
        0: "localssd",
        1: "beegfs",
        2: "lustre",
        3: "tmpfs",
        4: "nfs"
    }
    return mapping.get(store_code, "unknown")

def bytes_to_mb(file_size):
    """
    Convert a file size from bytes to megabytes (MB).
    
    Parameters:
    - file_size (str, int, or float): The file size in bytes (as an int/float) 
      or a string representation with size and unit (e.g., "1024 KiB").
      
    Returns:
    - float: The file size in MB.
    """
    # If file_size is a string, parse the value and unit
    if isinstance(file_size, str):
        size_num, size_unit = file_size.split()
        size_num = float(size_num)
        
        # Convert size to MB based on the unit
        if size_unit == "Bytes" or size_unit == "B":
            return size_num / (1024 ** 2)  # Convert bytes to MB
        elif size_unit == "KiB":
            return size_num / 1024  # Convert KiB to MB
        elif size_unit == "MiB":
            return size_num  # Already in MB
        elif size_unit == "GiB":
            return size_num * 1024  # Convert GiB to MB
        elif size_unit == "TiB":
            return size_num * (1024 ** 2)  # Convert TiB to MB
        else:
            raise ValueError(f"Unknown size unit: {size_unit}")
    elif isinstance(file_size, (int, float)):
        # If file_size is an integer or float, assume it's in bytes
        return file_size / (1024 ** 2)  # Convert bytes to MB
    else:
        raise TypeError("file_size must be a string or a number")


In [3]:
def is_sequential(numbers):
    if not numbers:  # Check if the list is empty
        return False

    sorted_numbers = sorted(numbers)  # Sort the numbers
    return all(sorted_numbers[i] + 1 == sorted_numbers[i + 1] for i in range(len(sorted_numbers) - 1))


def get_stat_file_pids(all_files):
    # Extract target tasks from blk_files
    target_tasks = set()
    for blk_file in all_files:
        # Get the filename without the path
        filename = os.path.basename(blk_file)
        # repalce ".local" for now
        filename = filename.replace(".local", "")
        
        # Split filename by '.'
        parts = filename.split('.')
        # print(f"get_stat_file_pids() : parts = {parts}")
        if len(parts) >= 3:
            # Get the target task from the -3 extension
            task = parts[-3]
            target_tasks.add(task)
    target_tasks = sorted(target_tasks)
    return target_tasks

import os
import glob
import json

import os
import json

def add_stat_to_df(trial_folder, monitor_timer_stat_io, 
                   operation, fname, task_pid, store_code):
    # Process the file name
    fname = fname.replace(".local", ".")
    fileName = ".".join(fname.split(".")[:-4])  # Remove last 4 extensions
    fileName = os.path.basename(fileName)      # Keep only the basename
    
    if monitor_timer_stat_io[1] == 0:
        monitor_timer_stat_io[1] = 1 # at least 1 operation if has I/O size
        # print(f"Error: opCount is 0 for task_pid[{task_pid}] fileName[{fileName}]")
    
    # Initialize statistics
    tmp_write_stat = {
        'aggregateFilesizeMB': bytes_to_mb(monitor_timer_stat_io[2]),
        'transferSize': monitor_timer_stat_io[2] / monitor_timer_stat_io[1],
        'operation': int(operation),
        'totalTime': monitor_timer_stat_io[0],
        'trMiB': bytes_to_mb(monitor_timer_stat_io[2] / monitor_timer_stat_io[0]),
        'storageType': store_code,
        'opCount': monitor_timer_stat_io[1],
        'taskPID': task_pid,
        'fileName': fileName,
    }
    print(f"fileName = {fileName}")
    
    if tmp_write_stat['totalTime'] > 100:
        print(f"Recorded large totalTime[{monitor_timer_stat_io}] from task_pid[{task_pid}] fileName[{fileName}]")
    
    if "6818-dc111" in task_pid:
        print(f"monitor_timer_stat_io: {monitor_timer_stat_io}")
    
    # Determine operation type
    op = "w" if operation == 0 else "r"
    
    # Ensure the trial folder exists
    if not os.path.exists(trial_folder):
        print(f"Trial folder does not exist: {trial_folder}")
        return tmp_write_stat
    
    # List all files in the trial folder
    all_files = os.listdir(trial_folder)
    # print(f"Total files in {trial_folder}: {len(all_files)}")
    
    # Find matching files using substring matching
    matching_files = [
        os.path.join(trial_folder, file)
        for file in all_files
        if f"{fileName}.{task_pid}.local.{op}" in file
    ]
    
    # if len(matching_files) == 0:
    #     print(f"No matching files found for fileName[{fileName}] task_pid[{task_pid}] op[{op}]")
    # else:
    #     print(f"Found {len(matching_files)} matching files: {matching_files}")
    
    # Process matching files to determine write pattern
    write_pattern = 0  # 0: seq, 1: rand
    for matching_file in matching_files:
        try:
            with open(matching_file) as f:
                w_blk_trace_data = json.load(f)
                blk_list = w_blk_trace_data.get('io_blk_range', [])
                
                # Validate blk_list length
                if len(blk_list) >= 4:
                    if blk_list[3] == -2:
                        write_pattern = 1
                        print(f"Detected random write pattern in file: {matching_file}")
                        break
                else:
                    print(f"Warning: Invalid 'io_blk_range' in file: {matching_file}")
        except Exception as e:
            print(f"Error processing file {matching_file}: {e}")
    
    # Update statistics
    tmp_write_stat['randomOffset'] = write_pattern
    
    return tmp_write_stat



In [4]:
target_tasks = ["python"] # omit srun from 1kgenome run

all_wf_df = pd.DataFrame(columns=WF_PARAMS)

# Find task PID's input and output to match script name
def get_wf_result_df(tests, WF_PARAMS, target_tasks, storageType="localssd"):
    wf_df = pd.DataFrame(columns=WF_PARAMS)

    # Identify trial folders
    wf_trial_folders = [
        folder for folder in glob.glob(f"{tests}/*")
        if folder.endswith(("t1", "t2", "t3"))
    ]
    print(f"Trial folders: {wf_trial_folders}")

    store_code = transform_store_code(storageType)

    for trial_folder in wf_trial_folders:
        blk_files = glob.glob(f"{trial_folder}/*_blk_trace.json")
        datalife_monitor = glob.glob(f"{trial_folder}/*.datalife.json")
        target_tasks = get_stat_file_pids(blk_files)
        print(f"blk_files count: {len(blk_files)}")
        print(f"datalife_monitor count: {len(datalife_monitor)}")
        print(f"target_tasks: {target_tasks}")

        for datalife_json in datalife_monitor:
            task_pid = os.path.basename(datalife_json).split(".")[1]
            if task_pid not in target_tasks:
                # print(f"Task PID[{task_pid}] not in target_tasks")
                continue

            try:
                with open(datalife_json) as f:
                    datalife_data = json.load(f)
            except json.JSONDecodeError:
                print(f"Error loading file: {datalife_json}")
                continue

            task_name = list(datalife_data.keys())[0]
            monitor_timer_stat = datalife_data[task_name]['monitor']
            system_timer_stat = datalife_data[task_name]['system']
            monitor_timer_targets = ["read", "write"]

            for fname in [f for f in blk_files if f".{task_pid}." in f]:
                op_type = "read" if ".r_blk_trace." in fname else "write"
                monitor_stat = monitor_timer_stat[op_type]
                
                # Extra check if monitor_stat has zero value, use value from the other op_type
                for i in range(len(monitor_stat)):
                    if monitor_stat[i] == 0:
                        if op_type == "read":
                            monitor_stat[i] = monitor_timer_stat["write"][i]
                        else:
                            monitor_stat[i] = monitor_timer_stat["read"][i]

                # if bytes_to_mb(monitor_stat[2]) == 0:
                #     print(f"No {op_type} stat for task_name[{task_name}] task_pid[{task_pid}]")
                #     continue
                
                # taskParallelism = task_name_to_parallelism[task_name]
                tmp_stat = add_stat_to_df(
                    trial_folder, monitor_stat, 
                    1 if op_type == "read" else 0,
                    fname, task_pid, store_code
                )
                wf_df = wf_df._append(tmp_stat, ignore_index=True)

    return wf_df

def get_test_folder_dfs(test_folder, WF_PARAMS, target_tasks,
                        storageType="localssd"):
    folder_dfs = pd.DataFrame(columns=WF_PARAMS)

    for tests in test_folder:
        stat_path = f"{exp_data_path}/{tests}"

        # Generate workflow data
        wf_df = get_wf_result_df(stat_path, WF_PARAMS, target_tasks,
                                 storageType=storageType)
        print(wf_df.head(5))
        print(f"df shape: {wf_df.shape}")

        # Append workflow data to the folder dataframe
        folder_dfs = folder_dfs._append(wf_df, ignore_index=True)
        
    return folder_dfs

wf_pfs_df = pd.DataFrame(columns=WF_PARAMS)



wf_pfs_df = wf_pfs_df._append(get_test_folder_dfs(test_folders, 
                                        WF_PARAMS, target_tasks,
                                        storageType="pfs"), ignore_index=True)



NameError: name 'pd' is not defined

In [ ]:
wf_pfs_df.shape, wf_pfs_df.columns

((89, 17),
 Index(['operation', 'randomOffset', 'transferSize', 'aggregateFilesizeMB',
        'numTasks', 'parallelism', 'totalTime', 'numNodesList', 'numNodes',
        'tasksPerNode', 'trMiB', 'storageType', 'opCount', 'taskName',
        'taskPID', 'fileName', 'stageOrder'],
       dtype='object'))

In [ ]:
def match_script_name(tests):
    # Find folders ending with [t1, t2, t3] in the test_folders
    test_folders = glob.glob(f"{tests}/*")
    wf_trial_folders = [folder for folder in test_folders if folder.endswith("t1") or folder.endswith("t2") or folder.endswith("t3")]
    print(f"Trial folders: {wf_trial_folders}")

    pid_input_output_dict = {}

    for trial_folder in wf_trial_folders:
        blk_files = glob.glob(f"{trial_folder}/*_blk_trace.json")
        print(f"len(blk_files) = {len(blk_files)}")
        unique_pids = get_stat_file_pids(blk_files)

        for pid in unique_pids:
            if pid not in pid_input_output_dict:
                pid_input_output_dict[pid] = {
                    "input": [],
                    "output": [],
                    "prevTask": "",
                    "taskName": ""
                }

            # Find the blk_trace_jsons files with the current task_pid
            w_blk_trace_jsons = glob.glob(f"{trial_folder}/*.{pid}.local.w_blk_trace.json")
            r_blk_trace_jsons = glob.glob(f"{trial_folder}/*.{pid}.local.r_blk_trace.json")

            # Replace ".local" with an empty string
            w_blk_trace_jsons = [f.replace(".local", "") for f in w_blk_trace_jsons]
            r_blk_trace_jsons = [f.replace(".local", "") for f in r_blk_trace_jsons]

            # Process write (output) files
            for w_file_path in w_blk_trace_jsons:
                w_file_name_parts = w_file_path.split(".")
                w_file_name = '.'.join(w_file_name_parts[:-3])  # Remove the last 3 extensions
                w_file_basename = os.path.basename(w_file_name)
                pid_input_output_dict[pid]['output'].append(w_file_basename)

            # Process read (input) files
            for r_file_path in r_blk_trace_jsons:
                r_file_name_parts = r_file_path.split(".")
                r_file_name = '.'.join(r_file_name_parts[:-3])  # Remove the last 3 extensions
                r_file_basename = os.path.basename(r_file_name)
                if r_file_basename not in pid_input_output_dict[pid]['input']:
                    pid_input_output_dict[pid]['input'].append(r_file_basename)

    return pid_input_output_dict

def get_wf_pid_script_dict(test_folder):

    all_wf_dict= {}

    for tests in test_folder:
        # # check of test folder starts with seq or par
        # if tests.startswith("seq"):
        #     numTasksWrite = 1
        #     numTasksRead = 1
        # else:
        #     numTasksWrite = 1
        #     numTasksRead = 1

        # io_size_dfs
        wf_dict = match_script_name(f"{exp_data_path}/{tests}")

        # # corr_matrix(wf_df, storageType)
        all_wf_dict.update(wf_dict)
    return all_wf_dict


all_wf_dict = get_wf_pid_script_dict(test_folders)

print(all_wf_dict)



Trial folders: ['./ddmd/ddmd_4n_pfs_large/4n_pfs_t1']
len(blk_files) = 89
{'143674-dlt05': {'input': [], 'output': ['stage0000_task0007.dcd', 'stage0000_task0007.h5'], 'prevTask': '', 'taskName': ''}, '143675-dlt05': {'input': [], 'output': ['stage0000_task0008.dcd', 'stage0000_task0008.h5'], 'prevTask': '', 'taskName': ''}, '143676-dlt05': {'input': [], 'output': ['stage0000_task0006.dcd', 'stage0000_task0006.h5'], 'prevTask': '', 'taskName': ''}, '170276-dlt04': {'input': [], 'output': ['stage0000_task0005.dcd', 'stage0000_task0005.h5'], 'prevTask': '', 'taskName': ''}, '170277-dlt04': {'input': [], 'output': ['stage0000_task0003.dcd', 'stage0000_task0003.h5'], 'prevTask': '', 'taskName': ''}, '170278-dlt04': {'input': [], 'output': ['stage0000_task0004.dcd', 'stage0000_task0004.h5'], 'prevTask': '', 'taskName': ''}, '190075-dlt02': {'input': [], 'output': ['stage0000_task0000.h5', 'stage0000_task0000.dcd'], 'prevTask': '', 'taskName': ''}, '190099-dlt02': {'input': [], 'output': ['s

In [ ]:
# Add prevTask column
wf_pfs_df['prevTask'] = ""
wf_pfs_df['taskName'] = "unknown"
print(wf_pfs_df.head(5))
print(wf_pfs_df.shape)

  operation randomOffset  transferSize  aggregateFilesizeMB numTasks  \
0         0            1   1066.397658             7.293896      NaN   
1         0            1   1066.397658             7.293896      NaN   
2         1            1    354.246967            17.099243      NaN   
3         1            1    354.246967            17.099243      NaN   
4         1            1    354.246967            17.099243      NaN   

  parallelism  totalTime numNodesList numNodes tasksPerNode       trMiB  \
0         NaN   0.026276          NaN      NaN          NaN  277.589578   
1         NaN   0.026276          NaN      NaN          NaN  277.589578   
2         NaN   0.211424          NaN      NaN          NaN   80.876723   
3         NaN   0.211424          NaN      NaN          NaN   80.876723   
4         NaN   0.211424          NaN      NaN          NaN   80.876723   

  storageType opCount taskName       taskPID                fileName  \
0           5    7172  unknown  190075-dlt02

In [ ]:
# save to initial df
wf_pfs_df.to_csv(f'./analysis_data/first_df.csv', index=False)

In [ ]:
import re
        
def matches_pattern(file_path, patterns):
    """Match a file path against task definition patterns."""
    file_name = os.path.basename(file_path)
    for pattern in patterns:
        try:
            regex_pattern = re.compile(pattern)
            if regex_pattern.fullmatch(file_name):
                return True
        except re.error as e:
            print(f"Invalid regex: {pattern}, Error: {e}")
    return False

def assign_task_names(tasks, task_order_dict):
    """Assign task names and predecessors to tasks based on patterns."""
    for task_pid, details in tasks.items():
        input_paths = details.get('input', [])
        output_paths = details.get('output', [])
        task_name = details.get('taskName', 'unknown')  # Use existing or default to 'unknown'

        # Iterate through each task definition
        for task, definition in task_order_dict.items():
            # Check if any output matches
            if any(matches_pattern(op, definition['outputs']) for op in output_paths):
                task_name = task
                tasks[task_pid]['taskName'] = task_name
                tasks[task_pid]['stage_order'] = definition['stage_order']
                break

            # If no output matches, check for input matches
            for prevTask, predecessor_def in definition['predecessors'].items():
                if any(matches_pattern(ip, predecessor_def.get('inputs', [])) for ip in input_paths):
                    task_name = task
                    tasks[task_pid]['taskName'] = task_name
                    tasks[task_pid]['stage_order'] = definition['stage_order']
                    tasks[task_pid]['prevTask'] = prevTask
                    # print(f"Input match found: Task [{task}] prevTask [{prevTask}] with input_patterns {predecessor_def.get('inputs', [])}")
                    break

        # If no valid match, warn about the task
        if task_name == 'unknown':
            print(f"Warning: Task PID {task_pid} could not be assigned a valid taskName.")

    return tasks
        
# Load task ordering json file
task_order_dict = {}
with open(f"{exp_data_path}/{SCRIPT_ORDER}.json") as f:
    task_order_dict = json.load(f)

print(f"task_order_dict : {task_order_dict}")
# Create a mapping from taskName to parallelism
task_name_to_parallelism = {task: info['parallelism'] for task, info in task_order_dict.items()}
task_name_to_num_tasks = {task: info['num_tasks'] for task, info in task_order_dict.items()}
print(task_name_to_parallelism)
print(task_name_to_num_tasks)



# Fill in task names
assign_task_names(all_wf_dict, task_order_dict)


    
# Unique list of taskNames
taskNames = set([v['taskName'] for v in all_wf_dict.values()])
print(f"Unique taskNames: {taskNames}")
print(f"all_wf_dict:")
for k,v in all_wf_dict.items():
    print(f"{k}:{v}")
print(f"all_wf_dict-----")


# if CURR_WF == "seismology":
#     # fill all empty taskName in wf_pfs_df as "siftSTFByMisfit"
#     wf_pfs_df.loc[wf_pfs_df['taskName'] == "", 'taskName'] = "siftSTFByMisfit"
    

if DEBUG:
    print(wf_pfs_df['fileName'].unique())
    print(wf_pfs_df['taskName'].unique())

task_order_dict : {'openmm': {'stage_order': 0, 'parallelism': 12, 'num_tasks': 12, 'predecessors': {'initial_data': {'inputs': []}}, 'outputs': ['stage\\d{4}_task\\d{4}\\.dcd', 'stage\\d{4}_task\\d{4}\\.h5']}, 'aggregate': {'stage_order': 1, 'parallelism': 1, 'num_tasks': 1, 'predecessors': {'openmm': {'inputs': ['stage\\d{4}_task\\d{4}\\.h5']}}, 'outputs': ['aggregated.h5']}, 'training': {'stage_order': 1, 'parallelism': 1, 'num_tasks': 1, 'predecessors': {'openmm': {'inputs': ['stage\\d{4}_task\\d{4}\\.h5']}, 'aggregate': {'inputs': ['aggregated.h5']}}, 'outputs': ['virtual_stage0000+_task[0-9]+\\.h5', 'embeddings-epoch-[0-9]+-[0-9]{8}-[0-9]{6}\\.h5', 'epoch-[0-9]+-[0-9]{8}-[0-9]{6}\\.pt', 'generator-weights\\.pt', 'encoder-weights\\.pt', 'discriminator-weights\\.pt']}, 'inference': {'stage_order': 2, 'parallelism': 1, 'num_tasks': 1, 'predecessors': {'openmm': {'inputs': ['stage\\d{4}_task\\d{4}\\.h5']}}, 'outputs': ['virtual_stage0003+_task[0-9]+\\.h5']}}
{'openmm': 12, 'aggregate

In [ ]:
# Create a mapping from taskPID to taskName
task_pid_to_name = {pid: info['taskName'] for pid, info in all_wf_dict.items()}
# print(task_pid_to_name)
# Update the DataFrame with the taskName
wf_pfs_df['taskName'] = wf_pfs_df['taskPID'].map(task_pid_to_name).fillna('unknown')

# Create a mapping from taskPID to prevTask
task_pid_to_prod_task = {pid: info['prevTask'] for pid, info in all_wf_dict.items()}
# print(task_pid_to_prod_task)
# add prevTask column to the DataFrame
wf_pfs_df['prevTask'] = wf_pfs_df['taskPID'].map(task_pid_to_prod_task).fillna('unknown')

# Print the updated DataFrame
# print(wf_pfs_df.head(5))
print(wf_pfs_df.shape)
df_unknown = wf_pfs_df[wf_pfs_df['taskName'] == 'unknown']
print(f"df unknown ({df_unknown.shape}):\n{df_unknown.head(5)}")
# print(f"df found:\n{wf_pfs_df[wf_pfs_df['taskName'] != 'unknown']}")

for pid, info in all_wf_dict.items():
    if 'stage_order' not in info:
        print(f"Missing 'stage_order' for taskPID: {pid}, info: {info}")
        if CURR_WF == "seismology":
            info['stage_order'] = 1
            info['taskName'] = "sG1IterDecon"

# Create a mapping from taskPID to stage_order
task_pid_to_stage_order = {pid: info['stage_order'] for pid, info in all_wf_dict.items()}
# print(task_pid_to_stage_order)
# add prevTask column to the DataFrame
wf_pfs_df['stageOrder'] = wf_pfs_df['taskPID'].map(task_pid_to_stage_order).fillna('-1')


# remove rows with filename contaiing string "SIFT.chr*.vcf"
for chrom in range(0, 11):
    wf_pfs_df = wf_pfs_df[~wf_pfs_df['fileName'].str.contains(f"SIFT.chr{chrom}.vcf")]
    
# Adjust dataframe prevTask
for index, row in wf_pfs_df.iterrows():
    if row['operation'] == 0:
        if row['taskName'] == '':
            wf_pfs_df.at[index, 'taskName'] = 'none'
    else:
        # Adjust read task predecessors
        taskName = row['taskName']
        fileName = row['fileName']
        if CURR_WF == "seismology":
            if taskName == '':
                # # Update taskName for read tasks to "sG1IterDecon"
                # wf_pfs_df.at[index, 'taskName'] = 'sG1IterDecon'
                taskName = 'sG1IterDecon'
                wf_pfs_df.at[index, 'taskName'] = 'sG1IterDecon'
        task_definition = task_order_dict[taskName]

        for task, inputs in task_definition['predecessors'].items():
            input_patterns = inputs['inputs']
            if matches_pattern(fileName, input_patterns):
                wf_pfs_df.at[index, 'prevTask'] = task



(89, 18)
df unknown ((0, 18)):
Empty DataFrame
Columns: [operation, randomOffset, transferSize, aggregateFilesizeMB, numTasks, parallelism, totalTime, numNodesList, numNodes, tasksPerNode, trMiB, storageType, opCount, taskName, taskPID, fileName, stageOrder, prevTask]
Index: []


In [ ]:
# print(f"df found:\n{wf_pfs_df[wf_pfs_df['taskName'] == 'trackstats']}")


# # Below values can only be updated once task name and per task parallelism is known
import math

# Assuming 'wf_pfs_df' is the DataFrame with a 'taskName' column
for index, row in wf_pfs_df.iterrows():
    task_name = row['taskName']
    if task_name in task_name_to_parallelism:
        task_parallelism = task_name_to_parallelism[task_name]
        task_num_tasks = task_name_to_num_tasks[task_name]

        
        # row['numNodesList'] = NUM_NODES_LIST
        # row['numTasks'] = task_num_tasks
        # row['tasksPerNode'] = math.ceil(task_parallelism / NUM_NODES_LIST)

        # Update the DataFrame
        wf_pfs_df.at[index, 'numNodesList'] = NUM_NODES_LIST #row['numNodesList']
        wf_pfs_df.at[index, 'numTasks'] = task_num_tasks
        # wf_pfs_df.at[index, 'tasksPerNode'] = math.ceil(task_parallelism / NUM_NODES_LIST)
        wf_pfs_df.at[index, 'parallelism'] = task_parallelism

if DEBUG:
    print(wf_pfs_df.shape)
    print(wf_pfs_df.head())

# # Remove row with nan data on any of the columns
# wf_pfs_df = wf_pfs_df.dropna()
print(wf_pfs_df.shape)

(89, 18)
  operation randomOffset  transferSize  aggregateFilesizeMB numTasks  \
0         0            1   1066.397658             7.293896       12   
1         0            1   1066.397658             7.293896       12   
2         1            1    354.246967            17.099243        1   
3         1            1    354.246967            17.099243        1   
4         1            1    354.246967            17.099243        1   

  parallelism  totalTime numNodesList numNodes tasksPerNode       trMiB  \
0          12   0.026276    [1, 2, 4]      NaN          NaN  277.589578   
1          12   0.026276    [1, 2, 4]      NaN          NaN  277.589578   
2           1   0.211424    [1, 2, 4]      NaN          NaN   80.876723   
3           1   0.211424    [1, 2, 4]      NaN          NaN   80.876723   
4           1   0.211424    [1, 2, 4]      NaN          NaN   80.876723   

  storageType opCount   taskName       taskPID                fileName  \
0           5    7172     openmm 

In [ ]:
# Expand the dataframe for multi-nodes configuration calculation
def expand_df(wf_pfs_df):

    # Create a new DataFrame to store updated rows
    updated_rows = []

    # Iterate through each row in the DataFrame
    for index, row in wf_pfs_df.iterrows():
        num_nodes_list = row['numNodesList']  # Extract the list of numNodes
        print(f"num_nodes_list: {num_nodes_list}")
        for num_nodes in num_nodes_list:
            # Create a copy of the current row
            new_row = row.copy()
            
            # Update the numNodes and tasksPerNode for the new row
            tasksPerNode = math.ceil(row['parallelism'] / num_nodes)
            new_row['tasksPerNode'] = tasksPerNode
            new_row['numNodes'] = num_nodes
            
            
            # Append the updated row to the list
            updated_rows.append(new_row)

    # Create a new DataFrame with the updated rows
    expanded_df = pd.DataFrame(updated_rows)

    # Reset the index of the expanded DataFrame
    expanded_df.reset_index(drop=True, inplace=True)

    # # Print the updated DataFrame for verification
    # print(expanded_df.shape)
    # print(expanded_df.head())
    
    return expanded_df


if MULTI_NODES:
    wf_pfs_df = expand_df(wf_pfs_df)
    # Print the updated DataFrame for verification
    if DEBUG:
        print(wf_pfs_df.shape)
        print(wf_pfs_df.head())

# for rows when parallelism is 1, update numNodes to 1
for index, row in wf_pfs_df.iterrows():
    if row['parallelism'] == 1:
        wf_pfs_df.at[index, 'numNodes'] = 1

num_nodes_list: [1, 2, 4]
num_nodes_list: [1, 2, 4]
num_nodes_list: [1, 2, 4]
num_nodes_list: [1, 2, 4]
num_nodes_list: [1, 2, 4]
num_nodes_list: [1, 2, 4]
num_nodes_list: [1, 2, 4]
num_nodes_list: [1, 2, 4]
num_nodes_list: [1, 2, 4]
num_nodes_list: [1, 2, 4]
num_nodes_list: [1, 2, 4]
num_nodes_list: [1, 2, 4]
num_nodes_list: [1, 2, 4]
num_nodes_list: [1, 2, 4]
num_nodes_list: [1, 2, 4]
num_nodes_list: [1, 2, 4]
num_nodes_list: [1, 2, 4]
num_nodes_list: [1, 2, 4]
num_nodes_list: [1, 2, 4]
num_nodes_list: [1, 2, 4]
num_nodes_list: [1, 2, 4]
num_nodes_list: [1, 2, 4]
num_nodes_list: [1, 2, 4]
num_nodes_list: [1, 2, 4]
num_nodes_list: [1, 2, 4]
num_nodes_list: [1, 2, 4]
num_nodes_list: [1, 2, 4]
num_nodes_list: [1, 2, 4]
num_nodes_list: [1, 2, 4]
num_nodes_list: [1, 2, 4]
num_nodes_list: [1, 2, 4]
num_nodes_list: [1, 2, 4]
num_nodes_list: [1, 2, 4]
num_nodes_list: [1, 2, 4]
num_nodes_list: [1, 2, 4]
num_nodes_list: [1, 2, 4]
num_nodes_list: [1, 2, 4]
num_nodes_list: [1, 2, 4]
num_nodes_li

In [ ]:
# Step 1: Rename the original column to preserve it
wf_pfs_df = wf_pfs_df.rename(columns={"aggregateFilesizeMB": "aggregateFilesizeMBtask"})

# Step 2: Group by taskName and numNodes, compute sum, then divide by numNodes
group_sums = (
    wf_pfs_df
    .groupby(["taskName", "numNodes"], as_index=False)["aggregateFilesizeMBtask"]
    .sum()
)

# Step 3: Compute the new aggregateFilesizeMB as sum / numNodes
group_sums["aggregateFilesizeMB"] = group_sums["aggregateFilesizeMBtask"] / group_sums["numNodes"]

# Step 4: Keep only the new column and keys for merging
group_sums = group_sums[["taskName", "numNodes", "aggregateFilesizeMB"]]

# Step 5: Merge back to the original dataframe
wf_pfs_df = wf_pfs_df.merge(group_sums, on=["taskName", "numNodes"], how="left")



wf_pfs_df.head(5)

,operation,randomOffset,transferSize,aggregateFilesizeMBtask,numTasks,parallelism,totalTime,numNodesList,numNodes,tasksPerNode,trMiB,storageType,opCount,taskName,taskPID,fileName,stageOrder,prevTask,aggregateFilesizeMB
0,0,1,1066.397658,7.293896,12,12,0.026276,"[1, 2, 4]",1,12,277.589578,5,7172,openmm,190075-dlt02,stage0000_task0000.h5,0,,174.608185
1,0,1,1066.397658,7.293896,12,12,0.026276,"[1, 2, 4]",2,6,277.589578,5,7172,openmm,190075-dlt02,stage0000_task0000.h5,0,,87.304092
2,0,1,1066.397658,7.293896,12,12,0.026276,"[1, 2, 4]",4,3,277.589578,5,7172,openmm,190075-dlt02,stage0000_task0000.h5,0,,43.652046
3,0,1,1066.397658,7.293896,12,12,0.026276,"[1, 2, 4]",1,12,277.589578,5,7172,openmm,190075-dlt02,stage0000_task0000.dcd,0,,174.608185
4,0,1,1066.397658,7.293896,12,12,0.026276,"[1, 2, 4]",2,6,277.589578,5,7172,openmm,190075-dlt02,stage0000_task0000.dcd,0,,87.304092


In [ ]:
# Print list of unique taskNames
wf_pfs_df['taskName'].unique()

array(['openmm', 'aggregate', 'training', 'inference'], dtype=object)

In [ ]:
# Save the updated DataFrame to a CSV file
wf_pfs_df.to_csv(f'./analysis_data/{test_folders[0]}.csv', index=False)

task_name_to_parallelism

{'openmm': 12, 'aggregate': 1, 'training': 1, 'inference': 1}

In [ ]:
# Calculate I/O time per taskName
write_sub_df = wf_pfs_df[wf_pfs_df['operation'] == 0]
read_sub_df = wf_pfs_df[wf_pfs_df['operation'] == 1]

task_io_time_total = wf_pfs_df.groupby('taskName')['totalTime'].sum()
task_io_time_write = write_sub_df.groupby('taskName')['totalTime'].sum()
task_io_time_read = read_sub_df.groupby('taskName')['totalTime'].sum()

task_io_time_adjust = {"read": 0, "write": 0}
total_wf_io_time = 0
total_wf_io_time_write = 0
total_wf_io_time_read = 0
print("Total I/O time per taskName:")
for task, write_time in task_io_time_write.items():
    # Adjust I/O time by parallelism
    write_time_adjusted = write_time / (task_name_to_parallelism[task] * len(NUM_NODES_LIST))
    task_io_time_adjust["write"]+=write_time_adjusted
    total_wf_io_time_write+=write_time_adjusted
    total_wf_io_time+=write_time_adjusted
    print(f" {task} (write): {write_time_adjusted} (sec)")
for task, read_time in task_io_time_read.items():
    # Adjust I/O time by parallelism
    read_time_adjusted = read_time / (task_name_to_parallelism[task] * len(NUM_NODES_LIST))
    task_io_time_adjust["read"]+=read_time_adjusted
    total_wf_io_time_read+=read_time_adjusted
    total_wf_io_time+=read_time_adjusted
    print(f" {task} (read): {read_time_adjusted} (sec)")
    
print(f"Total I/O time per workflow: {total_wf_io_time}")


# print("Total I/O time per stage:")
# for task, io_time in task_io_time_total.items():
#     # Adjust I/O time by parallelism
#     io_time_adjusted = io_time / task_name_to_parallelism[task]
#     task_io_time_adjust[task] = io_time_adjusted
#     total_wf_io_time+=io_time_adjusted
    
#     print(f" {task}: {io_time_adjusted} (sec)")

# # print(task_io_time_adjust)
# print(f"Total I/O time per workflow: {total_wf_io_time}")

Total I/O time per taskName:
 aggregate (write): 0.10176218799999999 (sec)
 inference (write): 2.8633e-05 (sec)
 openmm (write): 0.05025775266666667 (sec)
 training (write): 1.95473184 (sec)
 aggregate (read): 2.748506033 (sec)
 inference (read): 1.850840342 (sec)
 training (read): 11.551488481 (sec)
Total I/O time per workflow: 18.257615269666665


In [ ]:
# Read from file "./master_ior_df.csv"
df_ior = pd.read_csv("./master_ior_df.csv")
# df_ior = pd.read_csv(f'{test_folders[0]}.csv')
print(df_ior.columns)
print(df_ior.shape)

# oscache size is 25GiB
oscacheSizeMB = 25 * 1024  # Convert to MiB

Index(['Unnamed: 0', 'operation', 'randomOffset', 'transferSize',
       'aggregateFilesizeMB', 'numTasks', 'numNodes', 'tasksPerNode',
       'parallelism', 'totalTime_beegfs', 'totalTime_localssd',
       'totalTime_nfs', 'totalTime_tmpfs', 'trMiB_beegfs', 'trMiB_localssd',
       'trMiB_nfs', 'trMiB_tmpfs', 'trMiB_ave_beegfs', 'trMiB_ave_localssd',
       'trMiB_ave_nfs', 'trMiB_ave_tmpfs', 'selectStorage'],
      dtype='object')
(1740, 22)


In [ ]:

from sklearn.ensemble import GradientBoostingRegressor
import numpy as np

# Pre-sort df_ior 
if MULTI_NODES:
    df_ior_sorted = df_ior.sort_values(by=['operation', 'tasksPerNode', 'transferSize'])
else:
    df_ior_sorted = df_ior.sort_values(by=['operation', 'parallelism', 'transferSize'])
# Dictionary to store updates for each column
updates = {}

# # TODO: calculation of parallelism 60 and 30 for small file size has same tr

def calculate_4d_interpolation_with_extrapolation(df_ior_sorted, 
                                                  target_operation,
                                                  target_aggregateFilesizeMB, 
                                                  target_numNodes, 
                                                  target_parallelism, 
                                                  target_transfer_size,
                                                  par_col,
                                                  transferRate_column):
    """
    Perform 4D interpolation or extrapolation based on bounds for aggregateFilesizeMB, numNodes, parallelism, and transferSize.

    Parameters:
    - df: DataFrame containing the data.
    - target_operation: The operation to filter by.
    - target_aggregateFilesizeMB: Target aggregate file size (MB).
    - target_numNodes: Target number of nodes.
    - target_parallelism: Target parallelism value.
    - target_transfer_size: Target transfer size.
    - par_col: The column name representing parallelism (e.g., 'tasksPerNode' pr 'parallelism').
    - transferRate_column: Column name of the transfer rate values to interpolate.

    Returns:
    - Interpolated or extrapolated transfer rate value and the slope for transfer size interpolation.
    """
    
    df_ior_filtered = df_ior_sorted[df_ior_sorted['operation'] == target_operation].copy()
    
    def get_bounds(values, target):
        """
        Get bounds for interpolation or next two bounds when outside the range.
        """
        values = sorted(values)  # Ensure values are sorted
        values = np.array(values, dtype=np.float64)  # Convert to numpy array for compatibility

        if len(values) < 2:
            return (values[0], values[0]) if len(values) == 1 else (None, None)

        if target <= values[0]:
            return values[0], values[1]
        if target >= values[-1]:
            return values[-2], values[-1]

        lower = np.max(values[values < target], initial=None)
        upper = np.min(values[values >= target], initial=None)
        return lower, upper

    def calculate_interpolation(target_val, lower, upper, low_val, high_val):
        """
        Perform interpolation or extrapolation for a single dimension.
        Return the interpolated value, and the slope
        """
        if target_val < lower:
            # Extrapolate below lower bound
            slope = (high_val - low_val) / (upper - lower) if upper != lower else 0
            return low_val - slope * (lower - target_val), slope
        elif target_val > upper:
            # Extrapolate above upper bound
            slope = (high_val - low_val) / (upper - lower) if upper != lower else 0
            return high_val + slope * (target_val - upper), slope
        else:
            # Interpolate within bounds
            if upper > lower:
                slope = (target_val - lower) / (upper - lower)
                return low_val + slope * (high_val - low_val), slope
            else:
                return low_val, 1

    df_ior_filtered = df_ior_sorted[df_ior_sorted['operation'] == target_operation].copy()

    if df_ior_filtered.empty:
        raise ValueError("No rows found for the specified operation.")

    # Filter for numNodes # Not enough data for numnodes
    if MULTI_NODES:
        numNodes_values = sorted(df_ior_filtered['numNodes'].unique())
        # print(f"numNodes values: {numNodes_values}")
        lower_nodes, upper_nodes = get_bounds(numNodes_values, numNodes)
        # print(f"numNodes bounds for [{numNodes}]: {lower_nodes}, {upper_nodes}")
        df_ior_filtered = df_ior_filtered[df_ior_filtered['numNodes'].isin([lower_nodes, upper_nodes])]
    else:
        df_ior_filtered = df_ior_filtered[df_ior_filtered['numNodes'] == 1]

    # Parallelism bounds: tasksPerNode (when multi-nodes) or parallelism (when single node)
    parallelism_values = sorted(df_ior_filtered[par_col].unique())
    # print(f"Parallelism values: {parallelism_values}")
    lower_tasks, upper_tasks = get_bounds(parallelism_values, parallelism)
    # print(f"Parallelism bounds for [{parallelism}]: {lower_tasks}, {upper_tasks}")
    df_ior_filtered = df_ior_filtered[df_ior_filtered[par_col].isin([lower_tasks, upper_tasks])]
    
    # Transfer size bounds
    transfer_values = sorted(df_ior_filtered['transferSize'].unique())
    # print(f"Transfer values: {transfer_values}")
    lower_transfer, upper_transfer = get_bounds(transfer_values, transfer_size)
    # print(f"Transfer size bounds for [{transfer_size}]: {lower_transfer}, {upper_transfer}")
    df_ior_filtered = df_ior_filtered[df_ior_filtered['transferSize'].isin([lower_transfer, upper_transfer])]

    # Aggregate file size bounds
    aggregate_size_values = sorted(df_ior_filtered['aggregateFilesizeMB'].unique())
    # print(f"Aggregate size values: {aggregate_size_values}")
    lower_size, upper_size = get_bounds(aggregate_size_values, aggregateFilesizeMB)
    # print(f"Aggregate size bounds for [{aggregateFilesizeMB}]: {lower_size}, {upper_size}")
    df_ior_filtered = df_ior_filtered[df_ior_filtered['aggregateFilesizeMB'].isin([lower_size, upper_size])]
    
    # # get unique df
    # filtered_df = df_ior_filtered.drop_duplicates(subset=['operation', 'numNodes', 'tasksPerNode', 'transferSize', 'aggregateFilesizeMB'])
    filtered_df = df_ior_filtered

    # Get bounds for each dimension
    agg_values = filtered_df['aggregateFilesizeMB'].unique()
    node_values = filtered_df['numNodes'].unique()
    par_values = filtered_df[par_col].unique()
    ts_values = filtered_df['transferSize'].unique()

    agg_lower, agg_upper = get_bounds(agg_values, target_aggregateFilesizeMB)
    node_lower, node_upper = get_bounds(node_values, target_numNodes)
    par_lower, par_upper = get_bounds(par_values, target_parallelism)
    ts_lower, ts_upper = get_bounds(ts_values, target_transfer_size)

    # Initialize result
    estimated_trMiB_storage = 0.0
    total_weight = 0.0
    print("----------------------------")
    

    # For each dimension, find low and high transfer rates for interpolation
    # filtered_df = filtered_df.dropna(subset=[transferRate_column])
    agg_low_val = filtered_df.loc[filtered_df['aggregateFilesizeMB'] == agg_lower, transferRate_column].mean()
    agg_high_val = filtered_df.loc[filtered_df['aggregateFilesizeMB'] == agg_upper, transferRate_column].mean()
    if pd.isna(agg_high_val):
        agg_high_val = filtered_df.loc[filtered_df['aggregateFilesizeMB'] == agg_upper, transferRate_column].dropna().mean()
        print(f"NA : agg_high_val[{agg_high_val}]")
        print(f"{filtered_df.loc[filtered_df['aggregateFilesizeMB'] == agg_upper, transferRate_column].dropna()}")

    node_low_val = filtered_df.loc[filtered_df['numNodes'] == node_lower, transferRate_column].mean()
    node_high_val = filtered_df.loc[filtered_df['numNodes'] == node_upper, transferRate_column].mean()

    par_low_val = filtered_df.loc[filtered_df[par_col] == par_lower, transferRate_column].mean()
    par_high_val = filtered_df.loc[filtered_df[par_col] == par_upper, transferRate_column].mean()

    ts_low_val = filtered_df.loc[filtered_df['transferSize'] == ts_lower, transferRate_column].mean()
    ts_high_val = filtered_df.loc[filtered_df['transferSize'] == ts_upper, transferRate_column].mean()

    # Interpolate for each dimension
    totalSize_interpolated, agg_slope = calculate_interpolation(target_aggregateFilesizeMB, agg_lower, agg_upper, agg_low_val, agg_high_val)
    node_interpolated, node_slope = calculate_interpolation(target_numNodes, node_lower, node_upper, node_low_val, node_high_val)
    par_interpolated, par_slope = calculate_interpolation(target_parallelism, par_lower, par_upper, par_low_val, par_high_val)
    ts_interpolated, ts_slope = calculate_interpolation(target_transfer_size, ts_lower, ts_upper, ts_low_val, ts_high_val)

    # Combine weights
    combined_weight = 1.0  # Equal weight for now; modify as needed

    # Accumulate weighted interpolated values
    total_weight += combined_weight
    estimated_trMiB_storage += combined_weight * (totalSize_interpolated + node_interpolated + par_interpolated + ts_interpolated) / 4
    
    # Print estimated_trMiB_storage for the parallelism
    
    if math.isnan(agg_high_val):
        
    
        print(f'''target_aggregateFilesizeMB[{target_aggregateFilesizeMB}] 
            agg_lower[{agg_lower}] agg_upper[{agg_upper}] agg_low_val[{agg_low_val}] agg_high_val[{agg_high_val}]''')
        print(f'''
            filtered_df.loc[filtered_df['aggregateFilesizeMB'] == agg_upper, {transferRate_column}] 
            = {filtered_df.loc[filtered_df['aggregateFilesizeMB'] == agg_upper, transferRate_column]}
            ''')
        print(f'''target_numNodes[{target_numNodes}] target_parallelism[{target_parallelism}] : 
            estimated_trMiB_storage[{estimated_trMiB_storage}] 
            totalSize_interpolated[{totalSize_interpolated}]
            node_interpolated[{node_interpolated}]
            par_interpolated[{par_interpolated}]
            ts_interpolated[{ts_interpolated}]
            ''')

    # Normalize the result by total weight
    if total_weight > 0:
        estimated_trMiB_storage /= total_weight
    else:
        raise ValueError("No valid rows for interpolation or extrapolation. Check input bounds.")

    return estimated_trMiB_storage, ts_slope


# Ensure the required columns exist in the DataFrame
for storage in STORAGE_LIST:
    wf_pfs_df[f"estimated_trMiB_{storage}_p"] = None  # Initialize dynamically added columns

# Iterate through rows in wf_pfs_df
for index, row in wf_pfs_df.iterrows():  # Use `index` to directly update the DataFrame
    operation = row['operation']
    transfer_size = row['transferSize']
    aggregateFilesizeMB = row['aggregateFilesizeMB']
    numNodes = row['numNodes']
    if MULTI_NODES:
        task_parallelism = row['tasksPerNode']
        parallelism_range = [task_parallelism]
        # parallelism_range = [p for p in ALLOWED_PARALLELISM if p <= task_parallelism * 2]
    else:
        task_parallelism = row['parallelism']
    
        parallelism_range = [p for p in ALLOWED_PARALLELISM if p <= task_parallelism]        

    # Loop over storage options and calculate for each parallelism step
    for storage in STORAGE_LIST:
        for parallelism in parallelism_range:  # Loop over the calculated parallelism range                
            # Define the column name dynamically for the current parallelism level
            col_name_tr_storage = f"estimated_trMiB_{storage}_{parallelism}p"
            col_name_ts_slope = f"estimated_ts_slope_{storage}_{parallelism}p"
            
            try:
                if MULTI_NODES:
                    # Calculate the transfer rate
                    estimated_trMiB_storage, ts_slope = calculate_4d_interpolation_with_extrapolation(
                        df_ior_sorted,
                        operation, 
                        aggregateFilesizeMB, 
                        numNodes, 
                        parallelism, 
                        transfer_size,
                        'tasksPerNode',
                        f'trMiB_ave_{storage}'
                    )
                else:
                    # Calculate the transfer rate
                    estimated_trMiB_storage, ts_slope = calculate_4d_interpolation_with_extrapolation(
                        df_ior_sorted,
                        operation, 
                        aggregateFilesizeMB, 
                        numNodes, 
                        parallelism,
                        transfer_size,
                        'parallelism',
                        f'trMiB_ave_{storage}'
                    )
            except ValueError as e:
                print(f"Error calculating transfer rate for {storage} storage, parallelism {parallelism}: {e}")
                estimated_trMiB_storage = None

            # Check and add the column dynamically if it doesn't exist
            if col_name_tr_storage not in wf_pfs_df.columns:
                wf_pfs_df[col_name_tr_storage] = None  # Add the column with default None values
            # Add the calculated value to the DataFrame row
            wf_pfs_df.at[index, col_name_tr_storage] = estimated_trMiB_storage
            wf_pfs_df.at[index, col_name_ts_slope] = float(ts_slope)

            # # Debug print for confirmation
            print(f"Task[{row['taskName']}] Storage[{storage}] Parallelism[{parallelism}] aggregateFilesizeMB[{aggregateFilesizeMB}]-> {col_name_tr_storage} = {estimated_trMiB_storage}")


----------------------------
Task[openmm] Storage[localssd] Parallelism[12] aggregateFilesizeMB[174.60818481445312]-> estimated_trMiB_localssd_12p = 2338.3626384529293
----------------------------
Task[openmm] Storage[beegfs] Parallelism[12] aggregateFilesizeMB[174.60818481445312]-> estimated_trMiB_beegfs_12p = 6062.422728102541
----------------------------
Task[openmm] Storage[tmpfs] Parallelism[12] aggregateFilesizeMB[174.60818481445312]-> estimated_trMiB_tmpfs_12p = 24972.529615202377
----------------------------
Task[openmm] Storage[nfs] Parallelism[12] aggregateFilesizeMB[174.60818481445312]-> estimated_trMiB_nfs_12p = 437.48820268167935
----------------------------
Task[openmm] Storage[localssd] Parallelism[6] aggregateFilesizeMB[87.30409240722656]-> estimated_trMiB_localssd_6p = 1835.5208549084216
----------------------------
Task[openmm] Storage[beegfs] Parallelism[6] aggregateFilesizeMB[87.30409240722656]-> estimated_trMiB_beegfs_6p = 4177.676470022917
------------------------

In [ ]:
wf_pfs_df.head(5)

,operation,randomOffset,transferSize,aggregateFilesizeMBtask,numTasks,parallelism,totalTime,numNodesList,numNodes,tasksPerNode,...,estimated_trMiB_nfs_3p,estimated_ts_slope_nfs_3p,estimated_trMiB_localssd_1p,estimated_ts_slope_localssd_1p,estimated_trMiB_beegfs_1p,estimated_ts_slope_beegfs_1p,estimated_trMiB_tmpfs_1p,estimated_ts_slope_tmpfs_1p,estimated_trMiB_nfs_1p,estimated_ts_slope_nfs_1p
0,0,1,1066.397658,7.293896,12,12,0.026276,"[1, 2, 4]",1,12,...,None,NaN,None,NaN,None,NaN,None,NaN,None,NaN
1,0,1,1066.397658,7.293896,12,12,0.026276,"[1, 2, 4]",2,6,...,None,NaN,None,NaN,None,NaN,None,NaN,None,NaN
2,0,1,1066.397658,7.293896,12,12,0.026276,"[1, 2, 4]",4,3,...,366.375816,0.013801,None,NaN,None,NaN,None,NaN,None,NaN
3,0,1,1066.397658,7.293896,12,12,0.026276,"[1, 2, 4]",1,12,...,None,NaN,None,NaN,None,NaN,None,NaN,None,NaN
4,0,1,1066.397658,7.293896,12,12,0.026276,"[1, 2, 4]",2,6,...,None,NaN,None,NaN,None,NaN,None,NaN,None,NaN


In [ ]:
wf_pfs_df.columns

Index(['operation', 'randomOffset', 'transferSize', 'aggregateFilesizeMBtask',
       'numTasks', 'parallelism', 'totalTime', 'numNodesList', 'numNodes',
       'tasksPerNode', 'trMiB', 'storageType', 'opCount', 'taskName',
       'taskPID', 'fileName', 'stageOrder', 'prevTask', 'aggregateFilesizeMB',
       'estimated_trMiB_localssd_p', 'estimated_trMiB_beegfs_p',
       'estimated_trMiB_tmpfs_p', 'estimated_trMiB_nfs_p',
       'estimated_trMiB_localssd_12p', 'estimated_ts_slope_localssd_12p',
       'estimated_trMiB_beegfs_12p', 'estimated_ts_slope_beegfs_12p',
       'estimated_trMiB_tmpfs_12p', 'estimated_ts_slope_tmpfs_12p',
       'estimated_trMiB_nfs_12p', 'estimated_ts_slope_nfs_12p',
       'estimated_trMiB_localssd_6p', 'estimated_ts_slope_localssd_6p',
       'estimated_trMiB_beegfs_6p', 'estimated_ts_slope_beegfs_6p',
       'estimated_trMiB_tmpfs_6p', 'estimated_ts_slope_tmpfs_6p',
       'estimated_trMiB_nfs_6p', 'estimated_ts_slope_nfs_6p',
       'estimated_trMiB_local

In [ ]:
# Save updated DataFrame to a CSV file
wf_pfs_df.to_csv(f'./analysis_data/{test_folders[0]}_tr_estimated.csv', index=False)

# Split dataframe to different task names and save
unique_task_names = wf_pfs_df['taskName'].unique()

for task_name in unique_task_names:
    task_df = wf_pfs_df[wf_pfs_df['taskName'] == task_name].copy()
    
    write_df = task_df[task_df['operation'] == 0]
    read_df = task_df[task_df['operation'] == 1]    
    # Save the task DataFrame to CSV
    task_df.to_csv(f'./analysis_data/{test_folders[0]}_{task_name}_tr_estimated.csv', index=False)




In [ ]:
# Save one file with taskName == muttation_overlap and prevTask == individuals_merge with read operation only
mut_over_ind_merge_df = wf_pfs_df[(wf_pfs_df['taskName'] == 'mutation_overlap') & (wf_pfs_df['prevTask'] == 'individuals_merge') & (wf_pfs_df['operation'] == 1) ].copy()
# mutation_overlap_df.to_csv(f'./analysis_data/{test_folders[0]}_mutation_overlap_invidiauls_merge_tr_estimated.csv', index=False)

# Save one file with taskName == muttation_overlap and prevTask == sifting
mut_over_sift_df = wf_pfs_df[(wf_pfs_df['taskName'] == 'mutation_overlap') & (wf_pfs_df['prevTask'] == 'sifting') & (wf_pfs_df['operation'] == 1) ].copy()
# mutation_overlap_df.to_csv(f'./analysis_data/{test_folders[0]}_mutation_overlap_sifting_tr_estimated.csv', index=False)

# Save one file with taskName == muttation_overlap and prevTask == initial_Data
mut_over_initial_df = wf_pfs_df[(wf_pfs_df['taskName'] == 'mutation_overlap') & (wf_pfs_df['prevTask'] == 'initial_data') & (wf_pfs_df['operation'] == 1) ].copy()
# mutation_overlap_df.to_csv(f'./analysis_data/{test_folders[0]}_mutation_overlap_initial_data_tr_estimated.csv', index=False)


# Get the sum of aggregate file size for each df: mut_over_ind_merge_df, mut_over_sift_df, mut_over_initial_df
print(f"mut_over_ind_merge_df aggregateFilesizeMB sum: {mut_over_ind_merge_df['aggregateFilesizeMB'].sum()}")
print(f"mut_over_sift_df aggregateFilesizeMB sum: {mut_over_sift_df['aggregateFilesizeMB'].sum()}")
print(f"mut_over_initial_df aggregateFilesizeMB sum: {mut_over_initial_df['aggregateFilesizeMB'].sum()}")

# print the unique fileNames of mut_over_ind_merge_df
print(f"mut_over_ind_merge_df fileNames: {mut_over_ind_merge_df['fileName'].unique()}")

# Get one df of taskName == individuals_merge and fileName starts with "chr" and ends with ".tar.gz" and with write operation only
ind_merge_df = wf_pfs_df[(wf_pfs_df['taskName'] == 'individuals_merge') & (wf_pfs_df['fileName'].str.startswith("chr")) & (wf_pfs_df['operation'] == 0) ].copy()

# Get the sume of aggregate file size
print(f"ind_merge_df size: {ind_merge_df.shape}")
print(f"ind_merge_df aggregateFilesizeMB sum: {ind_merge_df['aggregateFilesizeMB'].sum()}")
# save ind_merge_df to csv
ind_merge_df.to_csv(f'./analysis_data/{test_folders[0]}_tmp_tr_estimated.csv', index=False)

mut_over_ind_merge_df aggregateFilesizeMB sum: 0.0
mut_over_sift_df aggregateFilesizeMB sum: 0.0
mut_over_initial_df aggregateFilesizeMB sum: 0.0
mut_over_ind_merge_df fileNames: []
ind_merge_df size: (0, 55)
ind_merge_df aggregateFilesizeMB sum: 0.0


In [ ]:
# Construct the SPM Here:
pc_df_list = []

task_name_list = list(wf_pfs_df['taskName'].unique())
print(f"Unique task names: {task_name_list}")
# task_name_list.append('none') # for first stage in the workflow

print(wf_pfs_df.columns)
print(wf_pfs_df.shape)

Unique task names: ['openmm', 'aggregate', 'training', 'inference']
Index(['operation', 'randomOffset', 'transferSize', 'aggregateFilesizeMBtask',
       'numTasks', 'parallelism', 'totalTime', 'numNodesList', 'numNodes',
       'tasksPerNode', 'trMiB', 'storageType', 'opCount', 'taskName',
       'taskPID', 'fileName', 'stageOrder', 'prevTask', 'aggregateFilesizeMB',
       'estimated_trMiB_localssd_p', 'estimated_trMiB_beegfs_p',
       'estimated_trMiB_tmpfs_p', 'estimated_trMiB_nfs_p',
       'estimated_trMiB_localssd_12p', 'estimated_ts_slope_localssd_12p',
       'estimated_trMiB_beegfs_12p', 'estimated_ts_slope_beegfs_12p',
       'estimated_trMiB_tmpfs_12p', 'estimated_ts_slope_tmpfs_12p',
       'estimated_trMiB_nfs_12p', 'estimated_ts_slope_nfs_12p',
       'estimated_trMiB_localssd_6p', 'estimated_ts_slope_localssd_6p',
       'estimated_trMiB_beegfs_6p', 'estimated_ts_slope_beegfs_6p',
       'estimated_trMiB_tmpfs_6p', 'estimated_ts_slope_tmpfs_6p',
       'estimated_trMiB

In [ ]:
# # Identify columns to normalize
# estimated_columns = [col for col in wf_pfs_df.columns if col.startswith('estimated_trMiB')]
# additional_columns = ['aggregateFilesizeMB', 'opCount']  # Add additional columns for normalization
# columns_to_normalize = estimated_columns + additional_columns

# # Calculate the global min and max across all selected columns
# global_min = wf_pfs_df[columns_to_normalize].min().min()
# global_max = wf_pfs_df[columns_to_normalize].max().max()

# # Normalize each column and store in new columns prefixed with 'norm_'
# for col in columns_to_normalize:
#     norm_col = f"norm_{col}"
#     wf_pfs_df[norm_col] = (wf_pfs_df[col] - global_min) / (global_max - global_min)
    
#     # Shift all normalized values by adding 1
#     wf_pfs_df[norm_col] += 1

# print(wf_pfs_df.columns)
# print(wf_pfs_df.shape)
# print(wf_pfs_df.head(5))

# Build Workflow Graph
Get max stage order for iteration
## First layers Nodes:
- stageOrder == 0
    - Add row as node attributes
    - Need node "order" 

## Middle layers
- Find previous layer nodes  (stageOrder -1)
 - Find if has the sane fileName, and make sure operation is 0 in stageOrder-1 and 1 in stageOrder
 - link edge, calculate IOI and SPM

## Final Layer
- When stageOrder reached max, stops

In [ ]:
import networkx as nx

# Initialize the workflow graph as a directed graph
WFG = nx.DiGraph()

# Get the unique and sorted stage orders
stage_order_list = sorted(int(x) for x in wf_pfs_df['stageOrder'].unique())
print(stage_order_list)

# Dictionary to store task nodes by stage and task name
stage_task_node_dict = {}


# Add nodes to the graph
for i, row in wf_pfs_df.iterrows():
    nodeName = f"{row['taskName']}:{row['taskPID']}:{row['fileName']}"  # Unique node identifier
    new_nodeData = row.to_dict()  # Node attributes as dictionary
    stageOrder = row['stageOrder']
    taskName = row['taskName']

    # if "openmm" in nodeName:
    #     print(f"Processing node: {nodeName}")

    # Check if the node already exists
    if WFG.has_node(nodeName):
        # Update node data by replacing NaN values with new valid values
        existing_nodeData = WFG.nodes[nodeName]
        for key, value in new_nodeData.items():
            if key in existing_nodeData:
                # Replace NaN with valid value if applicable
                if (
                    (existing_nodeData[key] is None or isinstance(existing_nodeData[key], float) and math.isnan(existing_nodeData[key])) 
                    and value is not None and not (isinstance(value, float) and math.isnan(value))
                ):
                    existing_nodeData[key] = value
            else:
                # Add the new key-value pair
                existing_nodeData[key] = value
        # Update the node data in the graph
        WFG.nodes[nodeName].update(existing_nodeData)
    else:
        # Add the node with attributes to the graph
        WFG.add_node(nodeName, **new_nodeData)

    # Populate the stage-task-node dictionary
    stage_task_node_dict.setdefault(stageOrder, {}).setdefault(taskName, []).append(nodeName)

# Show the number of nodes and edges in the graph
print(f"Number of nodes: {WFG.number_of_nodes()}")
print(f"First five nodes:")
for info in list(WFG.nodes(data=True))[:5]:
    print(info)

print(f"Number of edges: {WFG.number_of_edges()}")



[0, 1, 2]
Number of nodes: 84
First five nodes:
('openmm:190075-dlt02:stage0000_task0000.h5', {'operation': 0, 'randomOffset': 1, 'transferSize': 1066.3976575571667, 'aggregateFilesizeMBtask': 7.293895721435547, 'numTasks': 12, 'parallelism': 12, 'totalTime': 0.026275827, 'numNodesList': [1, 2, 4], 'numNodes': 1, 'tasksPerNode': 12, 'trMiB': 277.58957772996246, 'storageType': 5, 'opCount': 7172, 'taskName': 'openmm', 'taskPID': '190075-dlt02', 'fileName': 'stage0000_task0000.h5', 'stageOrder': 0, 'prevTask': '', 'aggregateFilesizeMB': 174.60818481445312, 'estimated_trMiB_localssd_p': None, 'estimated_trMiB_beegfs_p': None, 'estimated_trMiB_tmpfs_p': None, 'estimated_trMiB_nfs_p': None, 'estimated_trMiB_localssd_12p': 2338.3626384529293, 'estimated_ts_slope_localssd_12p': 0.013801320819390192, 'estimated_trMiB_beegfs_12p': 6062.422728102541, 'estimated_ts_slope_beegfs_12p': 0.013801320819390192, 'estimated_trMiB_tmpfs_12p': 24972.529615202377, 'estimated_ts_slope_tmpfs_12p': 0.013801320

In [ ]:
def add_producer_consumer_edge(WFG, prod_nodes, cons_nodes):
    """
    Add edges between producer and consumer nodes, calculating edge attributes for each storage type.
    """
    for taskName, prevNodeNames in prod_nodes.items():
        for prod_node_name in prevNodeNames:
            prod_fileName = WFG.nodes[prod_node_name]['fileName'].strip()
            
            max_op_count = WFG.nodes[prod_node_name]['opCount']

            for cons_task_name, currNodeNames in cons_nodes.items():
                for cons_node_name in currNodeNames:
                    if WFG.nodes[cons_node_name].get('prevTask') == taskName:
                        cons_fileName = WFG.nodes[cons_node_name]['fileName'].strip()

                        # Check if file names match
                        if prod_fileName == cons_fileName:
                            edge_attributes = {}

                            # Calculate edge attributes for each storage type
                            for storage in STORAGE_LIST:
                                prod_keys = [
                                    key for key in WFG.nodes[prod_node_name].keys()
                                    if key.startswith(f'estimated_trMiB_{storage}_')
                                ]
                                cons_keys = [
                                    key for key in WFG.nodes[cons_node_name].keys()
                                    if key.startswith(f'estimated_trMiB_{storage}_')
                                ]

                                for prod_key in prod_keys:
                                    try:
                                        n_prod = int(prod_key.split('_')[-1][:-1])
                                    except ValueError:
                                        continue

                                    prod_estimated_trMiB = WFG.nodes[prod_node_name].get(prod_key)
                                    if prod_estimated_trMiB is None or math.isnan(prod_estimated_trMiB):
                                        continue

                                    for cons_key in cons_keys:
                                        try:
                                            n_cons = int(cons_key.split('_')[-1][:-1])
                                        except ValueError:
                                            continue

                                        cons_estimated_trMiB = WFG.nodes[cons_node_name].get(cons_key)
                                        if cons_estimated_trMiB is None or math.isnan(cons_estimated_trMiB):
                                            continue

                                        prod_aggregateFilesizeMB = WFG.nodes[prod_node_name]['aggregateFilesizeMB']
                                        cons_aggregateFilesizeMB = WFG.nodes[cons_node_name]['aggregateFilesizeMB']
                                        prod_opCount = WFG.nodes[prod_node_name]['opCount']
                                        cons_opCount = WFG.nodes[cons_node_name]['opCount']
                                        
                                        # Get the attribute that starts with "estimated_ts_slope_{storage}_"
                                        prod_slope_key = [
                                            key for key in WFG.nodes[prod_node_name].keys()
                                            if key.startswith(f'estimated_ts_slope_{storage}_{n_prod}p')
                                        ][0]
                                        cons_slope_key = [
                                            key for key in WFG.nodes[cons_node_name].keys()
                                            if key.startswith(f'estimated_ts_slope_{storage}_{n_cons}p')
                                        ][0]
                                        prod_ts_slope = WFG.nodes[prod_node_name][prod_slope_key]
                                        cons_ts_slope = WFG.nodes[prod_node_name][cons_slope_key]
                                        
                                        # prod_op_weight = prod_opCount / (prod_opCount + cons_opCount)
                                        # cons_op_weight = cons_opCount / (prod_opCount + cons_opCount)
                                        # estT_prod = prod_opCount * prod_aggregateFilesizeMB / (prod_estimated_trMiB )
                                        # estT_cons = cons_opCount * cons_aggregateFilesizeMB / (cons_estimated_trMiB )
                                        
                                        # TS increases -> OP decrease => Perform Increase, less time
                                        # TS decrease -> OP increases => Perform Decrease, more time

                                        if prod_ts_slope > 0 :
                                            # If transfer_size has increased I/O affects, means it's becoming larger, then OP_count becomes smaller
                                            # but if OP count is large, means the transfer size is skwered with one large I/O operation
                                            # Thus penalize using OP count
                                            # but if OP count is small, it won't affect much
                                            estT_prod = prod_opCount * prod_aggregateFilesizeMB / prod_estimated_trMiB
                                        else:
                                            # If transfer size has decreased I/O affects, means it's becoming smaller, then operation count becomes larger
                                            # but if OP count is smal, means the transfer size is skwered with small total file size
                                            # Thus reward using OP count
                                            # but if OP count is large, it should have been reflected in the estimated_trMiB
                                            estT_prod = (1/prod_opCount) * prod_aggregateFilesizeMB / prod_estimated_trMiB
                                            
                                        if cons_ts_slope > 0:
                                            estT_cons = cons_opCount * cons_aggregateFilesizeMB / cons_estimated_trMiB 
                                        else:
                                            estT_cons = (1/cons_opCount) * cons_aggregateFilesizeMB / cons_estimated_trMiB
                                        
                                        SPM = estT_prod / estT_cons if estT_cons > 0 else float('inf')

                                        edge_attributes.update({
                                            f'estT_prod_{storage}_{n_prod}p': estT_prod,
                                            f'estT_cons_{storage}_{n_cons}p': estT_cons,
                                            f'SPM_{storage}_{n_prod}_{n_cons}p': SPM,
                                            'prod_aggregateFilesizeMB': prod_aggregateFilesizeMB,
                                            'cons_aggregateFilesizeMB': cons_aggregateFilesizeMB,
                                            'prod_max_parallelism': WFG.nodes[prod_node_name]['parallelism'],
                                            'cons_max_parallelism': WFG.nodes[cons_node_name]['parallelism'],
                                        })

                            # Add or update the edge in the graph
                            if WFG.has_edge(prod_node_name, cons_node_name):
                                WFG.edges[prod_node_name, cons_node_name].update(edge_attributes)
                            else:
                                WFG.add_edge(prod_node_name, cons_node_name, **edge_attributes)



def handle_initial_stage(WFG, cons_nodes):
    """
    Handle the initial stage where there is no producer, only consumers.
    Calculate and store `estT_cons` for each consumer node as node attributes
    with placeholders for producer-related values.
    """
    for cons_task_name, currNodeNames in cons_nodes.items():
        for cons_node_name in currNodeNames:
            # Update node attributes
            node_attributes = {}

            # Calculate node attributes for each storage type
            for storage in STORAGE_LIST:
                cons_keys = [
                    key for key in WFG.nodes[cons_node_name].keys()
                    if key.startswith(f'estimated_trMiB_{storage}_')
                ]

                for cons_key in cons_keys:
                    try:
                        # Extract the parallelism level from the key (e.g., estimated_trMiB_ssd_4p)
                        n_cons = int(cons_key.split('_')[-1][:-1])
                    except ValueError:
                        continue

                    cons_estimated_trMiB = WFG.nodes[cons_node_name].get(cons_key)

                    # Skip invalid or missing values
                    if cons_estimated_trMiB is None or math.isnan(cons_estimated_trMiB):
                        continue

                    cons_aggregateFilesizeMB = WFG.nodes[cons_node_name]['aggregateFilesizeMB']
                    cons_opCount = WFG.nodes[cons_node_name]['opCount']

                    # Calculate estimated task I/O time as intensity for consumer
                    estT_cons = (cons_opCount * cons_aggregateFilesizeMB) / cons_estimated_trMiB

                    # Store node attributes
                    node_attributes.update({
                        f'estT_prod_{storage}_0p': 0,  # Placeholder
                        f'estT_cons_{storage}_{n_cons}p': estT_cons,
                        'prod_aggregateFilesizeMB': 0,  # Placeholder
                        'cons_aggregateFilesizeMB': cons_aggregateFilesizeMB,
                        'prod_max_parallelism': 0,  # Placeholder
                        'cons_max_parallelism': WFG.nodes[cons_node_name]['parallelism'],
                    })

            # Update the node attributes in the graph
            WFG.nodes[cons_node_name].update(node_attributes)

for currOrder in stage_order_list:
    cons_nodes = stage_task_node_dict.get(currOrder, {})
    if currOrder == 0:
        # Handle stage 0
        if INITIAL_STAGE:
            handle_initial_stage(WFG, cons_nodes)
    else:
        # Add edges between producer and consumer nodes
        prevOrder = currOrder - 1
        prod_nodes = stage_task_node_dict.get(prevOrder, {})
        add_producer_consumer_edge(WFG, prod_nodes, cons_nodes)



In [ ]:
for u, v, attributes in WFG.edges(data=True):
    print(f"Edge ({u} -> {v}): Keys in attributes = {list(attributes.keys())}")

Edge (openmm:190075-dlt02:stage0000_task0000.h5 -> aggregate:190758-dlt02:stage0000_task0000.h5): Keys in attributes = ['estT_prod_localssd_12p', 'estT_cons_localssd_1p', 'SPM_localssd_12_1p', 'prod_aggregateFilesizeMB', 'cons_aggregateFilesizeMB', 'prod_max_parallelism', 'cons_max_parallelism', 'estT_prod_localssd_6p', 'SPM_localssd_6_1p', 'estT_prod_localssd_3p', 'SPM_localssd_3_1p', 'estT_prod_beegfs_12p', 'estT_cons_beegfs_1p', 'SPM_beegfs_12_1p', 'estT_prod_beegfs_6p', 'SPM_beegfs_6_1p', 'estT_prod_beegfs_3p', 'SPM_beegfs_3_1p', 'estT_prod_tmpfs_12p', 'estT_cons_tmpfs_1p', 'SPM_tmpfs_12_1p', 'estT_prod_tmpfs_6p', 'SPM_tmpfs_6_1p', 'estT_prod_tmpfs_3p', 'SPM_tmpfs_3_1p', 'estT_prod_nfs_12p', 'estT_cons_nfs_1p', 'SPM_nfs_12_1p', 'estT_prod_nfs_6p', 'SPM_nfs_6_1p', 'estT_prod_nfs_3p', 'SPM_nfs_3_1p']
Edge (openmm:74814-dlt06:stage0000_task0011.h5 -> aggregate:190758-dlt02:stage0000_task0011.h5): Keys in attributes = ['estT_prod_localssd_12p', 'estT_cons_localssd_1p', 'SPM_localssd_12

In [ ]:
def extract_SPM_estT_values(WFG):
    """
    Extract and store weighted SPM values for each producer-consumer pair, 
    including initial stage 0 nodes where the producer is 'initial_data'.
    
    Args:
        WFG (nx.DiGraph): A directed weighted graph with nodes and edges containing performance attributes.
    
    Returns:
        dict: Weighted SPM dictionary containing producer-consumer pairs, their SPM values, and task IO times.
    """
    SPM_estT_values = {}

    # Iterate through all edges in the graph
    for edge in WFG.edges(data=True):
        producer_node, consumer_node, attributes = edge
        prod_cons_pair = f"{WFG.nodes[producer_node]['taskName']}:{WFG.nodes[consumer_node]['taskName']}"
        
        if prod_cons_pair not in SPM_estT_values:
            SPM_estT_values[prod_cons_pair] = {
                'SPM': {},
                'estT_prod': {},
                'estT_cons': {},
                'rank': {},
                'par_prod': {},
                'par_cons': {},
                'dsize_prod': {},
                'dsize_cons': {},
            }
        
        for key, value in attributes.items():
            if key.startswith("SPM_"):
                storage_n = key.replace("SPM_", "")
                if storage_n not in SPM_estT_values[prod_cons_pair]['SPM']:
                    SPM_estT_values[prod_cons_pair]['SPM'][storage_n] = []
                SPM_estT_values[prod_cons_pair]['SPM'][storage_n].append(value)
            elif key.startswith("estT_prod_"):
                storage_n = key.replace("estT_prod_", "")
                if storage_n not in SPM_estT_values[prod_cons_pair]['estT_prod']:
                    SPM_estT_values[prod_cons_pair]['estT_prod'][storage_n] = []
                SPM_estT_values[prod_cons_pair]['estT_prod'][storage_n].append(value)
            elif key.startswith("estT_cons_"):
                storage_n = key.replace("estT_cons_", "")
                if storage_n not in SPM_estT_values[prod_cons_pair]['estT_cons']:
                    SPM_estT_values[prod_cons_pair]['estT_cons'][storage_n] = []
                SPM_estT_values[prod_cons_pair]['estT_cons'][storage_n].append(value)
            elif key == "prod_aggregateFilesizeMB":
                if "prod_aggregateFilesizeMB" not in SPM_estT_values[prod_cons_pair]['dsize_prod']:
                    SPM_estT_values[prod_cons_pair]['dsize_prod']['prod_aggregateFilesizeMB'] = []
                SPM_estT_values[prod_cons_pair]['dsize_prod']['prod_aggregateFilesizeMB'].append(value)
            elif key == "cons_aggregateFilesizeMB":
                if "cons_aggregateFilesizeMB" not in SPM_estT_values[prod_cons_pair]['dsize_cons']:
                    SPM_estT_values[prod_cons_pair]['dsize_cons']['cons_aggregateFilesizeMB'] = []
                SPM_estT_values[prod_cons_pair]['dsize_cons']['cons_aggregateFilesizeMB'].append(value)
            elif key == "prod_max_parallelism":
                if "prod_max_parallelism" not in SPM_estT_values[prod_cons_pair]['par_prod']:
                    SPM_estT_values[prod_cons_pair]['par_prod']['prod_max_parallelism'] = []
                SPM_estT_values[prod_cons_pair]['par_prod']['prod_max_parallelism'].append(value)
            elif key == "cons_max_parallelism":
                if "cons_max_parallelism" not in SPM_estT_values[prod_cons_pair]['par_cons']:
                    SPM_estT_values[prod_cons_pair]['par_cons']['cons_max_parallelism'] = []
                SPM_estT_values[prod_cons_pair]['par_cons']['cons_max_parallelism'].append(value)

    if INITIAL_STAGE:
        # Handle stage 0 nodes: Add as "initial_data" producer-consumer pairs
        for node, attributes in WFG.nodes(data=True):
            # print(f"Checking taskName: {attributes['taskName']} with stageOrder: {attributes['stageOrder']}") # WFG.in_degree(node) == 0 and 
            if attributes['stageOrder'] == 0 and attributes['operation'] == 1:
                prod_cons_pair = f"input:{attributes['taskName']}"
                if prod_cons_pair not in SPM_estT_values:
                    SPM_estT_values[prod_cons_pair] = {
                        'SPM': {},
                        'estT_prod': {},
                        'estT_cons': {},
                        'rank': {},
                        'par_prod': {},
                        'par_cons': {},
                        'dsize_prod': {},
                        'dsize_cons': {},
                    }

                for key, value in attributes.items():
                    if key.startswith("estT_cons_"):
                        storage_n = key.replace("estT_cons_", "")
                        if storage_n not in SPM_estT_values[prod_cons_pair]['estT_cons']:
                            SPM_estT_values[prod_cons_pair]['estT_cons'][storage_n] = []
                        SPM_estT_values[prod_cons_pair]['estT_cons'][storage_n].append(value)
                    elif key == "cons_aggregateFilesizeMB":
                        if "cons_aggregateFilesizeMB" not in SPM_estT_values[prod_cons_pair]['dsize_cons']:
                            SPM_estT_values[prod_cons_pair]['dsize_cons']['cons_aggregateFilesizeMB'] = []
                        SPM_estT_values[prod_cons_pair]['dsize_cons']['cons_aggregateFilesizeMB'].append(value)
                    elif key == "cons_max_parallelism":
                        if "cons_max_parallelism" not in SPM_estT_values[prod_cons_pair]['par_cons']:
                            SPM_estT_values[prod_cons_pair]['par_cons']['cons_max_parallelism'] = []
                        SPM_estT_values[prod_cons_pair]['par_cons']['cons_max_parallelism'].append(value)
                    elif key == "prod_aggregateFilesizeMB":
                        if "prod_aggregateFilesizeMB" not in SPM_estT_values[prod_cons_pair]['dsize_prod']:
                            SPM_estT_values[prod_cons_pair]['dsize_prod']['prod_aggregateFilesizeMB'] = []
                        SPM_estT_values[prod_cons_pair]['dsize_prod']['prod_aggregateFilesizeMB'].append(0)  # Placeholder
                    elif key == "prod_max_parallelism":
                        if "prod_max_parallelism" not in SPM_estT_values[prod_cons_pair]['par_prod']:
                            SPM_estT_values[prod_cons_pair]['par_prod']['prod_max_parallelism'] = []
                        SPM_estT_values[prod_cons_pair]['par_prod']['prod_max_parallelism'].append(0)  # Placeholder

    return SPM_estT_values

# Assuming WFG is the networkx graph with edge attributes
SPM_estT_values = extract_SPM_estT_values(WFG)





# # Print weighted SPM values for debugging
# for pair, data in SPM_estT_values.items():
#     print(f"\nProducer-Consumer Pair: {pair}")
#     print("SPM:")
#     for storage_n, spm_values in data['SPM'].items():
#         print(f"  {storage_n}: {spm_values[0:10]}")
#     print("estT_prod:")
#     for storage_n, estT_prod_values in data['estT_prod'].items():
#         print(f"  {storage_n}: {estT_prod_values[0:10]}")
#     print("estT_cons:")
#     for storage_n, estT_cons_values in data['estT_cons'].items():
#         print(f"  {storage_n}: {estT_cons_values[0:10]}")
#     print("dsize_prod:")
#     for storage_n, dsize_prod_values in data['dsize_prod'].items():
#         print(f"  {storage_n}: {dsize_prod_values[0:10]}")
#     print("dsize_cons:")
#     for storage_n, dsize_cons_values in data['dsize_cons'].items():
#         print(f"  {storage_n}: {dsize_cons_values[0:10]}")

In [ ]:
def normalize_estT_values_g(SPM_estT_values):
    """
    Normalizes estT_prod, estT_cons, and SPM values across all storage_n values globally.

    Parameters:
    SPM_estT_values (dict): Dictionary containing 'estT_prod', 'estT_cons', 'SPM', 'dsize_cons', and 'dsize_prod' values for normalization.

    Returns:
    dict: A new dictionary with globally normalized 'estT_prod', 'estT_cons', 'SPM', 'dsize_cons', and 'dsize_prod' values.
    """
    # Collect all values across the entire SPM_estT_values dictionary
    global_values = {key: [] for key in ['estT_prod', 'estT_cons', 'SPM', 'dsize_cons', 'dsize_prod']}
    for data in SPM_estT_values.values():
        for key in global_values.keys():
            global_values[key].extend(
                v for storage_n_values in data.get(key, {}).values() for v in storage_n_values
            )

    # Compute global min and max for each key
    global_min_max = {}
    for key, values in global_values.items():
        if values:
            global_min_max[key] = {'min': min(values), 'max': max(values)}
        else:
            global_min_max[key] = {'min': 0, 'max': 0}  # Default if no values

    # Create a new dictionary for normalized values
    normalized_SPM_estT_values = {}

    # Normalize each key within each producer-consumer pair
    for pair, data in SPM_estT_values.items():
        normalized_SPM_estT_values[pair] = {}
        for key in ['estT_prod', 'estT_cons', 'dsize_cons', 'dsize_prod']:
            min_val = global_min_max[key]['min']
            max_val = global_min_max[key]['max']

            normalized_SPM_estT_values[pair][key] = {}
            for storage_n, values in data.get(key, {}).items():
                if min_val == max_val:
                    normalized_values = [0.5] * len(values)  # If all values are the same
                else:
                    normalized_values = [(v - min_val) / (max_val - min_val) for v in values]
                
                normalized_SPM_estT_values[pair][key][storage_n] = normalized_values

        # Normalize SPM with specific logic
        key = 'SPM'
        min_val = global_min_max[key]['min']
        max_val = global_min_max[key]['max']

        normalized_SPM_estT_values[pair][key] = {}
        for storage_n, values in data.get(key, {}).items():
            normalized_values = []
            for v in values:
                if v > 1:
                    # Normalize values > 1 into [1, 2]
                    normalized_value = 1 + (v - 1) / (max_val - 1) if max_val > 1 else 1
                else:
                    # Keep values <= 1 as is
                    normalized_value = v
                normalized_values.append(normalized_value)
            
            normalized_SPM_estT_values[pair][key][storage_n] = normalized_values

    return normalized_SPM_estT_values


def normalize_estT_values(SPM_estT_values):
    """
    Normalizes estT_prod, estT_cons, and SPM values across storage_n values within each producer-consumer pair.

    Parameters:
    SPM_estT_values (dict): Dictionary containing 'estT_prod', 'estT_cons', 'SPM', 'dsize_cons', and 'dsize_prod' values for normalization.

    Returns:
    dict: A new dictionary with normalized 'estT_prod', 'estT_cons', 'SPM', 'dsize_cons', and 'dsize_prod' values within each pair.
    """
    print("Normalizing estT_prod, estT_cons, and SPM values within each producer-consumer pair.")
    # Create a new dictionary for normalized values
    normalized_SPM_estT_values = {}

    # Normalize each key within each producer-consumer pair
    for pair, data in SPM_estT_values.items():
        normalized_SPM_estT_values[pair] = {}

        # Compute pair-level min and max for each key
        pair_min_max = {}
        for key in ['estT_prod', 'estT_cons', 'SPM', 'dsize_cons', 'dsize_prod']:
            all_values = [
                v for storage_n_values in data.get(key, {}).values() for v in storage_n_values
            ]
            if all_values:
                pair_min_max[key] = {'min': min(all_values), 'max': max(all_values)}
            else:
                pair_min_max[key] = {'min': 0, 'max': 0}  # Default if no values

        # Normalize each key using pair-level min and max
        for key in ['estT_prod', 'estT_cons', 'dsize_cons', 'dsize_prod']:
            min_val = pair_min_max[key]['min']
            max_val = pair_min_max[key]['max']

            normalized_SPM_estT_values[pair][key] = {}
            for storage_n, values in data.get(key, {}).items():
                if min_val == max_val:
                    normalized_values = [0.5] * len(values)  # If all values are the same
                else:
                    normalized_values = [(v - min_val) / (max_val - min_val) for v in values]
                
                normalized_SPM_estT_values[pair][key][storage_n] = normalized_values

        # Normalize SPM with specific logic for each pair
        key = 'SPM'
        min_val = pair_min_max[key]['min']
        max_val = pair_min_max[key]['max']

        normalized_SPM_estT_values[pair][key] = {}
        for storage_n, values in data.get(key, {}).items():
            normalized_values = []
            for v in values:
                if v > 1:
                    # Normalize values > 1 into [1, 2]
                    normalized_value = 1 + (v - 1) / (max_val - 1) if max_val > 1 else 1
                else:
                    # Keep values <= 1 as is
                    normalized_value = v
                normalized_values.append(normalized_value)
            
            normalized_SPM_estT_values[pair][key][storage_n] = normalized_values

    return normalized_SPM_estT_values

# # Call the function to normalize estT_prod, estT_cons, and SPM globally
if NORMALIZE:
    SPM_estT_values = normalize_estT_values_g(SPM_estT_values)
    # SPM_estT_values = normalize_estT_values(SPM_estT_values)


# Print weighted SPM values for debugging
for pair, data in SPM_estT_values.items():
    print(f"\nProducer-Consumer Pair: {pair}")
    print("SPM:")
    for storage_n, spm_values in data['SPM'].items():
        print(f"  {storage_n}: {spm_values[0:10]} ...")
    print("estT_prod:")
    for storage_n, estT_prod_values in data['estT_prod'].items():
        print(f"  {storage_n}: {estT_prod_values[0:10]} ...")
    print("estT_cons:")
    for storage_n, estT_cons_values in data['estT_cons'].items():
        print(f"  {storage_n}: {estT_cons_values[0:10]} ...")
    print("dsize_prod:")
    for storage_n, dsize_prod_values in data['dsize_prod'].items():
        print(f"  {storage_n}: {dsize_prod_values[0:10]} ...")
    print("dsize_cons:")
    for storage_n, dsize_cons_values in data['dsize_cons'].items():
        print(f"  {storage_n}: {dsize_cons_values[0:10]} ...")


Producer-Consumer Pair: openmm:aggregate
SPM:
  localssd_12_1p: [88846876.34185123, 88803997.54861864, 88821222.59323013, 88726011.07842632, 88781715.5243901, 88770574.56492409, 88821222.59323013, 88815138.61334886, 88848562.01780947, 88832363.87003411] ...
  localssd_6_1p: [113186519.0338019, 113131013.73228192, 113153953.14222577, 113030511.68540742, 113102298.68195027, 113087941.21069042, 113153953.14222577, 113145371.31132096, 113188444.26373431, 113168310.93840976] ...
  localssd_3_1p: [113134221.12560631, 113078737.95017728, 113101670.7440849, 112978277.7347909, 113050034.85194851, 113035683.35666543, 113101670.7440849, 113093089.55309024, 113136144.57682702, 113116022.56384194] ...
  beegfs_12_1p: [12126336.053986272, 12119993.815781528, 12122899.112818424, 12108708.866159953, 12116769.539342994, 12115157.402658254, 12122899.112818424, 12121605.955534428, 12126442.380922284, 12124511.258753322] ...
  beegfs_6_1p: [17597096.335679717, 17587994.775501262, 17592095.41841883, 17571

In [ ]:
def calculate_averages_and_rank(SPM_estT_values):
    """
    Averages list values for estT_prod, estT_cons, SPM, and dsize, and calculates rank for each storage_n.
    
    rank = ave_dsize * (estT_prod + estT_cons) * abs(SPM - 1)
    
    Parameters:
    SPM_estT_values (dict): Dictionary containing 'estT_prod', 'estT_cons', 'SPM', and 'dsize' values.

    Returns:
    dict: Updated dictionary with averaged values and calculated ranks.
    """
    for pair, data in SPM_estT_values.items():
        print(f"\nAverages and time rank for Producer-Consumer Pair: {pair}")
        
        # Initialize a dictionary for ranks if not present
        if 'rank' not in data:
            data['rank'] = {}
        
        for storage_n in data['SPM'].keys():
            # Normalize storage_n to match keys in estT_prod and estT_cons
            prod_key = f"{'_'.join(storage_n.split('_')[:2])}p"  # Example: ssd_2_1p -> ssd_2p
            cons_key = f"{storage_n.split('_')[0]}_{storage_n.split('_')[2][:-1]}p"  # Example: ssd_2_1p -> ssd_1p
            prod_par = int(prod_key.split('_')[1].replace('p',''))
            cons_par = int(cons_key.split('_')[1].replace('p',''))
            
            # Average estT_prod
            estT_prod_values = data['estT_prod'].get(prod_key, [])
            avg_estT_prod = sum(estT_prod_values) / len(estT_prod_values) if estT_prod_values else 0.0
            # Average estT_cons
            estT_cons_values = data['estT_cons'].get(cons_key, [])
            avg_estT_cons = sum(estT_cons_values) / len(estT_cons_values) if estT_cons_values else 0.0
            # Average SPM
            spm_values = data['SPM'].get(storage_n, [])
            avg_spm = sum(spm_values) / len(spm_values) if spm_values else 0.0
            # Average dsize (corrected key access)
            dsize_prod_values = data['dsize_prod'].get("", [])  # Access dsize using the empty key as seen in the normalized output
            ave_prod_dsize = sum(dsize_prod_values) #/ len(dsize_prod_values) if dsize_prod_values else 1.0  # Default to 1.0 if no dsize
            dsize_cons_values = data['dsize_cons'].get("", [])  # Access dsize using the empty key as seen in the normalized output
            ave_cons_dsize = sum(dsize_cons_values) #/ len(dsize_cons_values) if dsize_cons_values else 1.0  # Default to 1.0 if no dsize
            
            if MULTI_NODES == False:
                prod_max_parallelism = data['par_prod'].get("", [])[0] # all should be same
                cons_max_parallelism = data['par_cons'].get("", [])[0] # all should be same
                prod_seq_tasks = prod_max_parallelism/prod_par
                cons_seq_tasks = cons_max_parallelism/cons_par
                # Calculate rank
                rank = (prod_seq_tasks * ave_prod_dsize * avg_estT_prod + cons_seq_tasks * cons_seq_tasks * avg_estT_cons)
            else:
                # prod_dsize_weight = ave_prod_dsize / (ave_prod_dsize + ave_cons_dsize)
                # cons_dsize_weight = ave_cons_dsize / (ave_prod_dsize + ave_cons_dsize)
                
                # rank = (prod_dsize_weight * avg_estT_prod  + cons_dsize_weight * avg_estT_cons )
                rank = avg_estT_prod  + avg_estT_cons 
            

            # Store averages back in the dictionary for reference
            data['estT_prod'][prod_key] = [avg_estT_prod]  # Replace list with a single averaged value
            data['estT_cons'][cons_key] = [avg_estT_cons]
            data['SPM'][storage_n] = [avg_spm]
            data['dsize_prod'][storage_n] = [ave_prod_dsize]
            data['dsize_cons'][storage_n] = [ave_cons_dsize]

            # Store the calculated rank
            data['rank'][storage_n] = [rank]

            # Print for debugging
            print(f"- {storage_n} time_rank: {rank}\n  - avg_estT_prod: {avg_estT_prod}\n  - ave_prod_dsize: {ave_prod_dsize}\n  - avg_estT_cons: {avg_estT_cons}\n  - ave_cons_dsize: {ave_cons_dsize}")

    return SPM_estT_values


def calculate_sums_and_rank(SPM_estT_values):
    """
    Sums list values for estT_prod, estT_cons, SPM, and dsize, and calculates rank for each storage_n.
        
    Parameters:
    SPM_estT_values (dict): Dictionary containing 'estT_prod', 'estT_cons', 'SPM', and 'dsize' values.

    Returns:
    dict: Updated dictionary with summed values and calculated ranks.
    """
    for pair, data in SPM_estT_values.items():
        print(f"\nSums and time rank for Producer-Consumer Pair: {pair}")
        
        # Initialize a dictionary for ranks if not present
        if 'rank' not in data:
            data['rank'] = {}
        
        # get total dsize of all producers and consumers
        total_dsize_prod = sum([sum(v) for k, v in data['dsize_prod'].items()])
        total_dsize_cons = sum([sum(v) for k, v in data['dsize_cons'].items()])
                
        # print(f"data['SPM'].keys() : {data['SPM'].keys()}")
        # print(f"data['estT_cons'].keys() : {data['estT_cons'].keys()}")
        
        if data['SPM'].keys():

            for storage_n in data['SPM'].keys():
                # Normalize storage_n to match keys in estT_prod and estT_cons
                prod_key = f"{'_'.join(storage_n.split('_')[:2])}p"  # Example: ssd_2_1p -> ssd_2p
                cons_key = f"{storage_n.split('_')[0]}_{storage_n.split('_')[2][:-1]}p"  # Example: ssd_2_1p -> ssd_1p
                prod_par = int(prod_key.split('_')[1].replace('p', ''))
                cons_par = int(cons_key.split('_')[1].replace('p', ''))
                
                # Sum estT_prod
                estT_prod_values = data['estT_prod'].get(prod_key, [])
                sum_estT_prod = sum(estT_prod_values) #if estT_prod_values else 0.0
                # Sum estT_cons
                estT_cons_values = data['estT_cons'].get(cons_key, [])
                sum_estT_cons = sum(estT_cons_values) #if estT_cons_values else 0.0
                # Average SPM
                spm_values = data['SPM'].get(storage_n, [])
                avg_spm = sum(spm_values) / len(spm_values) if spm_values else 0.0
                # Sum dsize (corrected key access)
                dsize_prod_values = data['dsize_prod'].get('prod_aggregateFilesizeMB', [])
                sum_prod_dsize = sum(dsize_prod_values) #if dsize_prod_values else 1.0  # Default to 1.0 if no dsize
                dsize_cons_values = data['dsize_cons'].get('cons_aggregateFilesizeMB', [])
                sum_cons_dsize = sum(dsize_cons_values) #if dsize_cons_values else 1.0  # Default to 1.0 if no dsize
                
                if MULTI_NODES == False:
                    prod_max_parallelism = data['par_prod'].get("", [])[0]  # all should be same
                    cons_max_parallelism = data['par_cons'].get("", [])[0]  # all should be same
                    prod_seq_tasks = prod_max_parallelism / prod_par
                    cons_seq_tasks = cons_max_parallelism / cons_par
                    # Calculate rank
                    rank = (prod_seq_tasks * sum_prod_dsize * sum_estT_prod +
                            cons_seq_tasks * sum_cons_dsize * sum_estT_cons)
                else:
                    # rank = (sum_prod_dsize * sum_estT_prod  + sum_cons_dsize * sum_estT_cons )
                    prod_time_weight = sum_prod_dsize / (sum_prod_dsize + sum_cons_dsize)
                    cons_time_weight = sum_cons_dsize / (sum_prod_dsize + sum_cons_dsize)
                    
                    # rank = prod_time_weight * sum_estT_prod  + cons_time_weight * sum_estT_cons
                    rank = prod_time_weight * sum_estT_prod + cons_time_weight * sum_estT_cons
                
                # Store sums back in the dictionary for reference
                data['estT_prod'][prod_key] = [sum_estT_prod]  # Replace list with a single summed value
                data['estT_cons'][cons_key] = [sum_estT_cons]
                data['SPM'][storage_n] = [avg_spm]  # Still using the average for SPM
                data['dsize_prod'][storage_n] = [sum_prod_dsize]
                data['dsize_cons'][storage_n] = [sum_cons_dsize]

                # Store the calculated rank
                data['rank'][storage_n] = [rank]

                # Print for debugging
                print(f"- {storage_n} time_rank: {rank}\n  - sum_estT_prod: {sum_estT_prod}\n  - sum_prod_dsize: {sum_prod_dsize}\n  - sum_estT_cons: {sum_estT_cons}\n  - sum_cons_dsize: {sum_cons_dsize}")
        else:
            if INITIAL_STAGE:
                for storage_n in data['estT_cons'].keys():
                    cons_key = f"{storage_n.split('_')[0]}_{storage_n.split('_')[1][:-1]}p"  # Example: ssd_2_1p -> ssd_1p
                    cons_par = int(cons_key.split('_')[1].replace('p', ''))
                    
                    # Sum estT_cons
                    estT_cons_values = data['estT_cons'].get(cons_key, [])
                    sum_estT_cons = sum(estT_cons_values) if estT_cons_values else 0.0
                    # Sum dsize (corrected key access)
                    dsize_cons_values = data['dsize_cons'].get("", [])
                    sum_cons_dsize = sum(dsize_cons_values) if dsize_cons_values else 1.0  # Default to 1.0 if no dsize
                    
                    if MULTI_NODES == False:
                        cons_max_parallelism = data['par_cons'].get("", [])[0]  # all should be same
                        cons_seq_tasks = cons_max_parallelism / cons_par
                        # Calculate rank
                        rank = (cons_seq_tasks * sum_cons_dsize * sum_estT_cons)
                    else:
                        rank = (sum_cons_dsize * sum_estT_cons)
                    
                    # Store sums back in the dictionary for reference
                    data['estT_cons'][cons_key] = [sum_estT_cons]
                    data['dsize_cons'][storage_n] = [sum_cons_dsize]

                    # Store the calculated rank
                    data['rank'][storage_n] = [rank]

                    # Print for debugging
                    print(f"- {storage_n} time_rank: {rank}\n  - sum_estT_prod: {sum_estT_prod}\n  - sum_prod_dsize: {sum_prod_dsize}\n  - sum_estT_cons: {sum_estT_cons}\n  - sum_cons_dsize: {sum_cons_dsize}")


    return SPM_estT_values


# combined_SPM_estT_values = calculate_averages_and_rank(SPM_estT_values)
combined_SPM_estT_values = calculate_sums_and_rank(SPM_estT_values)



Sums and time rank for Producer-Consumer Pair: openmm:aggregate
- localssd_12_1p time_rank: 1261.2392553331415
  - sum_estT_prod: 6422.745801976514
  - sum_prod_dsize: 2095.2982177734375
  - sum_estT_cons: 7.233227045671938e-05
  - sum_cons_dsize: 8574.817222595215
- localssd_6_1p time_rank: 1606.7451517371505
  - sum_estT_prod: 8182.203128235479
  - sum_prod_dsize: 2095.2982177734375
  - sum_estT_cons: 7.233227045671938e-05
  - sum_cons_dsize: 8574.817222595215
- localssd_3_1p time_rank: 1606.002709412805
  - sum_estT_prod: 8178.422308282795
  - sum_prod_dsize: 2095.2982177734375
  - sum_estT_cons: 7.233227045671938e-05
  - sum_cons_dsize: 8574.817222595215
- beegfs_12_1p time_rank: 486.460490628584
  - sum_estT_prod: 2477.2549297969445
  - sum_prod_dsize: 2095.2982177734375
  - sum_estT_cons: 0.00020441384569712062
  - sum_cons_dsize: 8574.817222595215
- beegfs_6_1p time_rank: 705.9292687647546
  - sum_estT_prod: 3594.879704354362
  - sum_prod_dsize: 2095.2982177734375
  - sum_estT_

In [ ]:
# This is not used anymore
def filter_storage_options(combined_SPM_estT_values, CURR_WF):
    """
    Filters combined_SPM_estT_values to keep only valid storage options based on the workflow (CURR_WF),
    but allows all beegfs storage options.

    Parameters:
    combined_SPM_estT_values (dict): Dictionary containing SPM, estT values, and rank.
    CURR_WF (str): Current workflow identifier.

    Returns:
    dict: Filtered dictionary containing only valid storage options.
    """
    valid_storage_options = {
        "1kg": {
            # "ssd": {"1": {"1p"}, "2": {"2p"}, "5": {"5p"}, 
            #         "150": {"5p"}, "60": {"2p"}, "30": {"1p"}, "20": {"1p"},
            # },
            # "beegfs": {"1": {"1p"}, "2": {"2p"}, "5": {"5p"}, 
            #            "150": {"5p"}, "60": {"2p"}, "30": {"1p"}, "20": {"1p"}
            # }
            "ssd": {"1": {"1p"}, "2": {"2p"}, "5": {"5"}, "4": {"4p"}, "10": {"10p"}, 
                    "150": {"5p", "10p"}, "60": {"2p", "4p"}, "30": {"1p", "2p"}, "20": {"1p", "2p"},
            },
            "beegfs": {"1": {"1p"}, "2": {"2p"}, "5": {"5"}, "4": {"4p"}, "10": {"10p"}, 
                    "150": {"5p", "10p"}, "60": {"2p", "4p"}, "30": {"1p", "2p"}, "20": {"1p", "2p"},
            }
        },
        "pyflex_s9_96f": {
            "ssd": {"1": {"1p","24p"}, "24":{"1p","24p"}},
        },
        "pyflex_240f": {
            "ssd": {"1": {"1p"}, "8":{"1p"}, "15": {"1p"},"30": {"1p"}, },
            "beegfs": {"1": {"1p"}, "8":{"1p"}, "15": {"1p"},"30": {"1p"},}
        },
        "ddmd_4n_l": {
            "localssd": {"6": {"1p", "4p"}, "4": {"1p", "4p"}, "3": {"1p", "4p"}, "2": {"1p", "4p"}, "1": {"1p", "4p"}},
            "tmpfs": {"6": {"1p", "4p"}, "4": {"1p", "4p"}, "3": {"1p", "4p"}, "2": {"1p", "4p"}, "1": {"1p", "4p"}},
            "beegfs": {"6": {"1p", "4p"}, "4": {"1p", "4p"}, "3": {"1p", "4p"}, "2": {"1p", "4p"}, "1": {"1p", "4p"}},
            "nfs": {"6": {"1p", "4p"}, "4": {"1p", "4p"}, "3": {"1p", "4p"}, "2": {"1p", "4p"}, "1": {"1p", "4p"}}
        },
        "ddmd_2n_s": {
            "ssd": {"6": {"1p", "4p"}, "4": {"1p", "4p"}, "3": {"1p", "4p"}, "2": {"1p", "4p"}, "1": {"1p", "4p"}},
            "beegfs": {"6": {"1p", "4p"}, "4": {"1p", "4p"}, "3": {"1p", "4p"}, "2": {"1p", "4p"}, "1": {"1p", "4p"}}
        }
    }

    # Check if CURR_WF has valid storage configurations
    if CURR_WF not in valid_storage_options:
        print(f"No valid storage configurations found for CURR_WF = {CURR_WF}.")
        return combined_SPM_estT_values

    filtered_results = {}
    for pair, data in combined_SPM_estT_values.items():
        filtered_rank = {}
        print(f"\nFiltering for Producer-Consumer Pair: {pair}")

        for storage_n, rank in data['rank'].items():
            storage_type = storage_n.split("_")[0]
            sp = storage_n.split("_")
            if sp[1] in valid_storage_options[CURR_WF].get(storage_type, {}) and sp[2] in valid_storage_options[CURR_WF][storage_type][sp[1]]:
                filtered_rank[storage_n] = rank
                print(f"  Keeping ({storage_type}): {storage_n} with Rank: {rank}")

            # if storage_type == "beegfs":
            #     # Apply filtering rules for ssd
            #     # Allow all beegfs storage options
            #     filtered_rank[storage_n] = rank
            # elif storage_type == "ssd":
            #     # Apply filtering rules for ssd
            #     if sp[1] in valid_storage_options[CURR_WF].get("ssd", {}) and sp[2] in valid_storage_options[CURR_WF]["ssd"][sp[1]]:
            #         filtered_rank[storage_n] = rank
            #         print(f"  Keeping (ssd): {storage_n} with Rank: {rank}")

        # Only add pair to results if we have valid storage options remaining
        if filtered_rank:
            filtered_results[pair] = {
                "SPM": data["SPM"],
                "estT_prod": data["estT_prod"],
                "estT_cons": data["estT_cons"],
                "dsize_prod": data["dsize_prod"],
                "dsize_cons": data["dsize_cons"],
                "rank": filtered_rank
            }
        else:
            print(f"  No valid storage options for {pair}, skipping...")

    return filtered_results


# Example of calling the method
filtered_combined_SPM_estT_values = filter_storage_options(combined_SPM_estT_values, CURR_WF)




Filtering for Producer-Consumer Pair: openmm:aggregate
  Keeping (localssd): localssd_6_1p with Rank: [1606.7451517371505]
  Keeping (localssd): localssd_3_1p with Rank: [1606.002709412805]
  Keeping (beegfs): beegfs_6_1p with Rank: [705.9292687647546]
  Keeping (beegfs): beegfs_3_1p with Rank: [706.2581560981841]
  Keeping (tmpfs): tmpfs_6_1p with Rank: [177.8578591504699]
  Keeping (tmpfs): tmpfs_3_1p with Rank: [177.88760008900655]
  Keeping (nfs): nfs_6_1p with Rank: [8048.6983369438]
  Keeping (nfs): nfs_3_1p with Rank: [8049.26432388279]

Filtering for Producer-Consumer Pair: openmm:training
  Keeping (localssd): localssd_6_1p with Rank: [78.36952910349592]
  Keeping (localssd): localssd_3_1p with Rank: [78.33331586106709]
  Keeping (beegfs): beegfs_6_1p with Rank: [34.432087430546524]
  Keeping (beegfs): beegfs_3_1p with Rank: [34.44812903018393]
  Keeping (tmpfs): tmpfs_6_1p with Rank: [8.675268390201962]
  Keeping (tmpfs): tmpfs_3_1p with Rank: [8.676719063229102]
  Keeping (

In [ ]:
def display_top_sorted_averaged_rank(combined_SPM_estT_values, baseline=0, top_n=5):
    """
    Displays the top N storage_n selections based on averaged rank values closest to the baseline.
    
    Args:
        combined_SPM_estT_values (dict): Dictionary containing averaged SPM and rank values.
        baseline (float): Baseline to calculate closeness for sorting.
        top_n (int): Number of top results to display.
    """
    print(f"Top {top_n} Averaged SPM Values Closest to Baseline = {baseline}:\n")

    for pair, data in combined_SPM_estT_values.items():
        producer, consumer = pair.split(":")
        print(f"Producer: {producer}, Consumer: {consumer}")
        
        # Collect and sort rank values by closeness to the baseline
        sorted_spm = sorted(
            data['rank'].items(),
            key=lambda item: abs(item[1][0] - baseline) if item[1] else float('inf')  # Handle empty rank values
        )
        
        top_n_displayed = 0
        
        for rank, (storage_n, avg_spm) in enumerate(sorted_spm, start=1):
            avg_spm_value = avg_spm[0] if avg_spm else float('inf')  # Extract the actual value
            print(f"- Rank {rank}: {storage_n} with Averaged rank = {avg_spm_value}")
            top_n_displayed += 1
            if top_n_displayed >= top_n:
                break
        print()  # Blank line for readability

# Call the function to display results
display_top_sorted_averaged_rank(filtered_combined_SPM_estT_values, top_n=40)

# TODO: save results for the wrong estimation to emphasize the benchmark parameter importance for number of node used on different storage, for correct estimation

Top 40 Averaged SPM Values Closest to Baseline = 0:

Producer: openmm, Consumer: aggregate
- Rank 1: tmpfs_6_1p with Averaged rank = 177.8578591504699
- Rank 2: tmpfs_3_1p with Averaged rank = 177.88760008900655
- Rank 3: beegfs_6_1p with Averaged rank = 705.9292687647546
- Rank 4: beegfs_3_1p with Averaged rank = 706.2581560981841
- Rank 5: localssd_3_1p with Averaged rank = 1606.002709412805
- Rank 6: localssd_6_1p with Averaged rank = 1606.7451517371505
- Rank 7: nfs_6_1p with Averaged rank = 8048.6983369438
- Rank 8: nfs_3_1p with Averaged rank = 8049.26432388279

Producer: openmm, Consumer: training
- Rank 1: tmpfs_6_1p with Averaged rank = 8.675268390201962
- Rank 2: tmpfs_3_1p with Averaged rank = 8.676719063229102
- Rank 3: beegfs_6_1p with Averaged rank = 34.432087430546524
- Rank 4: beegfs_3_1p with Averaged rank = 34.44812903018393
- Rank 5: localssd_3_1p with Averaged rank = 78.33331586106709
- Rank 6: localssd_6_1p with Averaged rank = 78.36952910349592
- Rank 7: nfs_6_1p 

In [ ]:
def select_best_storage_and_parallelism(combined_SPM_estT_values, baseline=0):
    """
    Selects the best storage type and parallelism level for each producer-consumer pair
    based on rank values and averages the rank of each storage type.

    Parameters:
    combined_SPM_estT_values (dict): Dictionary containing averaged SPM, estT values, and rank.
    baseline (float): Baseline for comparison.

    Returns:
    dict: Results containing the best storage type, parallelism, and averaged ranks.
    """
    print(f"Selecting the Best Storage Type and Parallelism Closest to Baseline = {baseline}:\n")
    
    results = {}  # Store the best storage, parallelism, and averaged ranks for each pair

    for pair, data in combined_SPM_estT_values.items():
        producer, consumer = pair.split(":")
        print(f"Producer: {producer}, Consumer: {consumer}")
        
        # Group rank values by storage type
        storage_groups = {}
        for storage_n, rank_values in data['rank'].items():
            storage_type = storage_n.split("_")[0]  # Extract storage type (e.g., 'beegfs' or 'ssd')
            if storage_type not in storage_groups:
                storage_groups[storage_type] = []
            storage_groups[storage_type].append((storage_n, rank_values[0]))  # Add rank values

        # Compute the average rank for each storage type
        averaged_storage_ranks = {}
        for storage_type, ranks in storage_groups.items():
            avg_rank = sum(rank for _, rank in ranks) / len(ranks) if ranks else float('inf')
            averaged_storage_ranks[storage_type] = avg_rank

        # Rank storage types by their average rank
        sorted_storages = sorted(averaged_storage_ranks.items(), key=lambda x: x[1])

        # Select the best storage type and its parallelism level
        best_storage_type = sorted_storages[0][0]
        best_storage_avg_rank = sorted_storages[0][1]

        # Find the best parallelism level within the best storage type
        best_parallelism, best_rank = min(storage_groups[best_storage_type], key=lambda x: x[1])

        # Store the results for this pair
        results[pair] = {
            "best_storage_type": best_storage_type,
            "best_parallelism": best_parallelism,
            "best_rank": best_rank,
            "avg_rank_by_storage": averaged_storage_ranks
        }

        # Display the results for this pair
        print(f"  Best Storage Type: {best_storage_type}")
        print(f"  Best Parallelism: {best_parallelism}")
        print(f"  Best Rank: {best_rank}")
        print("  Average Rank by Storage Type:")
        for storage, avg_rank in averaged_storage_ranks.items():
            print(f"    {storage}: {avg_rank}")
        print()

    return results


# Call the function to get results
best_results = select_best_storage_and_parallelism(combined_SPM_estT_values, baseline=0)

Selecting the Best Storage Type and Parallelism Closest to Baseline = 0:

Producer: openmm, Consumer: aggregate
  Best Storage Type: tmpfs
  Best Parallelism: tmpfs_12_1p
  Best Rank: 118.10577237599743
  Average Rank by Storage Type:
    localssd: 1491.329038827699
    beegfs: 632.8826384971743
    tmpfs: 157.9504105384913
    nfs: 7612.943974959199

Producer: openmm, Consumer: training
  Best Storage Type: tmpfs
  Best Parallelism: tmpfs_12_1p
  Best Rank: 5.760816203476638
  Average Rank by Storage Type:
    localssd: 72.7401082558238
    beegfs: 30.86922906504991
    tmpfs: 7.7042678856359
    nfs: 371.31929007448



Producer: individuals, Consumer: individuals_merge
- Rank 1: ssd_30_1p with Averaged rank = 0.017388674921199967
- Rank 2: ssd_150_5p with Averaged rank = 0.025450925433364482
- Rank 3: ssd_60_2p with Averaged rank = 0.027707793661594655
- Rank 4: beegfs_30_1p with Averaged rank = 0.11409868110395552
- Rank 5: beegfs_150_5p with Averaged rank = 0.15283971516043474
- Rank 6: beegfs_60_2p with Averaged rank = 0.1640937117000688

Producer: sifting, Consumer: frequency
- Rank 1: ssd_1_1p with Averaged rank = 0.007905239274441504
- Rank 2: ssd_5_5p with Averaged rank = 0.010924589075783754
- Rank 3: ssd_2_2p with Averaged rank = 0.011782311624048044
- Rank 4: beegfs_1_1p with Averaged rank = 0.03276770721034341
- Rank 5: beegfs_5_5p with Averaged rank = 0.04351388869274737
- Rank 6: beegfs_2_2p with Averaged rank = 0.047428847428693445

Producer: sifting, Consumer: mutation_overlap
- Rank 1: ssd_1_1p with Averaged rank = 0.006164074887066781
- Rank 2: ssd_5_5p with Averaged rank = 0.008987874902372775
- Rank 3: ssd_2_2p with Averaged rank = 0.008987874902372775
- Rank 4: beegfs_1_1p with Averaged rank = 0.03462791314274227
- Rank 5: beegfs_5_5p with Averaged rank = 0.04702673543023321
- Rank 6: beegfs_2_2p with Averaged rank = 0.04702673543023321

Producer: individuals_merge, Consumer: frequency
- Rank 1: ssd_1_1p with Averaged rank = 0.007260063117152319
- Rank 2: ssd_5_5p with Averaged rank = 0.009466438609848152
- Rank 3: ssd_2_2p with Averaged rank = 0.009836729412092633
- Rank 4: beegfs_1_1p with Averaged rank = 0.015507831177314671
- Rank 5: beegfs_5_5p with Averaged rank = 0.020602345215376867
- Rank 6: beegfs_2_2p with Averaged rank = 0.02229498046469096

Producer: individuals_merge, Consumer: mutation_overlap
- Rank 1: ssd_1_1p with Averaged rank = 0.0056576403798836656
- Rank 2: ssd_5_5p with Averaged rank = 0.007613622655703506
- Rank 3: ssd_2_2p with Averaged rank = 0.007613622655703506
- Rank 4: beegfs_1_1p with Averaged rank = 0.016779884264833043
- Rank 5: beegfs_5_5p with Averaged rank = 0.022766717957834724
- Rank 6: beegfs_2_2p with Averaged rank = 0.022766717957834724


# Pyflex
Producer: gettracks, Consumer: trackstats
- Rank 1: ssd_8_1p with Averaged rank = 8150.477574427682
- Rank 2: ssd_30_1p with Averaged rank = 8305.0662443393
- Rank 3: ssd_15_1p with Averaged rank = 8305.0662443393
- Rank 4: beegfs_30_1p with Averaged rank = 121927.46267373498
- Rank 5: beegfs_15_1p with Averaged rank = 121927.46267373498
- Rank 6: beegfs_8_1p with Averaged rank = 126452.05431477434

Producer: trackstats, Consumer: identifymcs
- Rank 1: beegfs_1_1p with Averaged rank = 873.6177783736396
- Rank 2: ssd_1_1p with Averaged rank = 1122.166010497233

# DDMD
Producer: openmm, Consumer: aggregate
- Rank 1: beegfs_6_1p with Averaged rank = 38.01161496516639
- Rank 2: beegfs_3_1p with Averaged rank = 40.94923739460553
- Rank 3: ssd_6_1p with Averaged rank = 91.21511150536406
- Rank 4: ssd_3_1p with Averaged rank = 94.65189593765005

Producer: openmm, Consumer: inference
- Rank 1: beegfs_6_1p with Averaged rank = 42.355713343587986
- Rank 2: beegfs_3_1p with Averaged rank = 45.62905787394207
- Rank 3: ssd_6_1p with Averaged rank = 101.63950226886205
- Rank 4: ssd_3_1p with Averaged rank = 105.46905483854921

Producer: openmm, Consumer: training
- Rank 1: beegfs_6_1p with Averaged rank = 15.442315179256221
- Rank 2: beegfs_3_1p with Averaged rank = 16.635791066736406
- Rank 3: ssd_6_1p with Averaged rank = 37.05636967689609
- Rank 4: ssd_3_1p with Averaged rank = 38.45259675043185